# YOLOv5s on D-Fire: training, held-out evaluation and speed benchmark

This notebook trains **YOLOv5s** on the corrected D-Fire split (classes: `smoke`, `fire`), evaluates it on the
held-out test set, and benchmarks inference speed against the YOLOv4 model (`yolov4_darknet_train.ipynb`, `yolov4_test_evaluation.ipynb`).
It is part of the project *Design and Performance Evaluation of a Real-Time Vision-Based Fire and Smoke
Detection System Using Deep Learning*.

**Environment:** Google Colab, Tesla T4 GPU, project stored on Google Drive.

| Setting | Value |
|---|---|
| Model | YOLOv5s, pretrained `yolov5s.pt` |
| Input size | 416 x 416 |
| Batch size / epochs | 16 / 85 |
| Split (images) | 13 777 train / 3 444 val / 4 306 test |
| Test-set objects | 5 189 (2 311 smoke, 2 878 fire) |

## Results on the held-out test set

| Metric | YOLOv4 | YOLOv5s |
|---|---|---|
| mAP@0.5 | 0.746 | 0.760 |
| mAP@0.5:0.95 | not measured | 0.417 |
| Precision (conf 0.25, IoU 0.50) | 0.689 | 0.673 |
| Recall (conf 0.25, IoU 0.50) | 0.738 | 0.763 |
| F1 (conf 0.25, IoU 0.50) | 0.713 | 0.715 |
| Best F1 (at confidence) | 0.731 (0.40) | 0.746 (0.45) |
| Mean latency, T4, batch 1 | 27.87 ms | 8.30 ms |
| FPS from mean latency | 35.9 | 120.5 |

YOLOv4 values come from `yolov4_test_evaluation.ipynb`. Precision, recall and F1 use the same matching code
for both models (standardised in that notebook). Latency covers the model forward pass only (preprocessing and
NMS excluded), measured on the same 200 test images for both models.

## Expected Drive layout

```
Fire_smoke_detection/
├── data/processed/DFireDataset_split/{images,labels}/{train,val,test}
├── external/YOLOv5/            # created by section 1.3
├── configs/YOLOv5/             # DFire_corrected.yaml is written here
└── experiments/{YOLOv5,YOLOv4,comparison}/
```

## How to use

Every cell defines its own paths, so after a Colab disconnect you only need to re-run section 1 and then the
cell you want. Section 1.3 is first-time setup only. Section 2 depends on files from the earlier, uncorrected run. Sections 7.3 and 9 depend on files written by `yolov4_test_evaluation.ipynb`.

## 1. Setup

### 1.1 Mount Google Drive and verify the project

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import os

# --------------------------------------------------
# 1. Define the Google Drive mountpoint
# --------------------------------------------------

MOUNTPOINT = Path("/content/drive")

print("===== DRIVE STATUS BEFORE MOUNTING =====")
print("Mountpoint exists:", MOUNTPOINT.exists())
print(
    "Is active Google Drive mount:",
    os.path.ismount(MOUNTPOINT),
)


# --------------------------------------------------
# 2. Preserve conflicting local contents
# --------------------------------------------------
#
# If /content/drive is not an active mount but has
# files inside it, those are temporary local files.
# Renaming preserves them and frees the mountpoint.
# --------------------------------------------------

local_backup = None

if (
    MOUNTPOINT.exists()
    and not os.path.ismount(MOUNTPOINT)
):
    local_entries = list(MOUNTPOINT.iterdir())

    print(
        "Non-mounted local entries:",
        len(local_entries),
    )

    if local_entries:
        timestamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

        local_backup = Path(
            "/content/"
            f"drive_unmounted_backup_{timestamp}"
        )

        MOUNTPOINT.rename(local_backup)

        print(
            "Preserved local folder as:",
            local_backup,
        )


# --------------------------------------------------
# 3. Create an empty mountpoint and mount Drive
# --------------------------------------------------

MOUNTPOINT.mkdir(
    parents=True,
    exist_ok=True,
)

drive.mount(
    str(MOUNTPOINT),
    force_remount=False,
)


# --------------------------------------------------
# 4. Verify the permanent project
# --------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/Fire_smoke_detection"
)

DATASET = (
    PROJECT
    / "data"
    / "processed"
    / "DFireDataset_split"
)

YOLOV4_SUMMARY = (
    PROJECT
    / "experiments"
    / "YOLOv4"
    / "results"
    / "yolov4_test_summary.csv"
)

print("\n===== PROJECT VERIFICATION =====")
print(
    "Drive is active mount:",
    os.path.ismount(MOUNTPOINT),
)
print("Project exists:", PROJECT.exists())
print("Dataset exists:", DATASET.exists())
print(
    "YOLOv4 summary exists:",
    YOLOV4_SUMMARY.exists(),
)

===== DRIVE STATUS BEFORE MOUNTING =====
Mountpoint exists: False
Is active Google Drive mount: False
Mounted at /content/drive

===== PROJECT VERIFICATION =====
Drive is active mount: True
Project exists: True
Dataset exists: True
YOLOv4 summary exists: True


### 1.2 Verify the corrected split counts (optional)

Counts images and labels per split. Slow on Drive, so run it only when the dataset has changed. Depends on the cell above.

In [ ]:
# --------------------------------------------------
# 5. Count the corrected YOLO-format data
# --------------------------------------------------

expected_counts = {
    "train": 13777,
    "val": 3444,
    "test": 4306,
}

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
}

all_counts_correct = True

print("\n===== CORRECTED DATASET COUNTS =====")

for split, expected in expected_counts.items():
    image_folder = DATASET / "images" / split
    label_folder = DATASET / "labels" / split

    images = [
        file
        for file in image_folder.iterdir()
        if (
            file.is_file()
            and file.suffix.lower()
            in image_extensions
        )
    ] if image_folder.exists() else []

    labels = (
        list(label_folder.glob("*.txt"))
        if label_folder.exists()
        else []
    )

    correct = (
        len(images) == expected
        and len(labels) == expected
    )

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Expected:", expected)
    print("Counts correct:", correct)

    if not correct:
        all_counts_correct = False


# --------------------------------------------------
# 6. Final interpretation
# --------------------------------------------------

print("\n===== FINAL RESULT =====")

if (
    os.path.ismount(MOUNTPOINT)
    and PROJECT.exists()
    and DATASET.exists()
    and all_counts_correct
):
    print(
        "SUCCESS: Google Drive is mounted and "
        "the corrected dataset is available."
    )
else:
    print(
        "STOP: Drive or dataset verification failed."
    )

if local_backup:
    print(
        "Temporary local backup retained at:",
        local_backup,
    )

### 1.3 Clone YOLOv5 and write the model config (first-time setup)

Run once on a fresh Drive. It clones `ultralytics/yolov5` into `external/YOLOv5`, writes
`models/custom_yolov5s.yaml` (the standard `yolov5s.yaml` with `nc: 2`) and downloads the pretrained
`yolov5s.pt`. Anything that already exists is left untouched, so it is safe to re-run.

This cell was added when the notebook was prepared for publication. The reported run used a YOLOv5 copy
already on Drive, so there is no saved output here.

In [ ]:
from pathlib import Path
import subprocess
import urllib.request

# --------------------------------------------------
# 1. Define the paths
# --------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/Fire_smoke_detection"
)

YOLOV5_REPO = (
    PROJECT
    / "external"
    / "YOLOv5"
)

CUSTOM_MODEL_YAML = (
    YOLOV5_REPO
    / "models"
    / "custom_yolov5s.yaml"
)

PRETRAINED_WEIGHTS = (
    YOLOV5_REPO
    / "yolov5s.pt"
)

YOLOV5_URL = "https://github.com/ultralytics/yolov5.git"

WEIGHTS_URL = (
    "https://github.com/ultralytics/yolov5/"
    "releases/download/v7.0/yolov5s.pt"
)

if not PROJECT.exists():
    raise FileNotFoundError(
        "The project folder was not found. "
        "Mount Google Drive first (section 1.1)."
    )


# --------------------------------------------------
# 2. Clone YOLOv5 if it is not on Drive yet
# --------------------------------------------------
#
# An existing repository is never modified.
# --------------------------------------------------

print("===== YOLOv5 REPOSITORY =====")

if (YOLOV5_REPO / "train.py").exists():
    print("Already present:", YOLOV5_REPO)

elif YOLOV5_REPO.exists() and any(YOLOV5_REPO.iterdir()):
    raise RuntimeError(
        "The YOLOv5 folder exists but has no train.py. "
        "Inspect it before cloning."
    )

else:
    YOLOV5_REPO.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            YOLOV5_URL,
            str(YOLOV5_REPO),
        ],
        check=True,
    )

    print("Cloned into:", YOLOV5_REPO)


# --------------------------------------------------
# 3. Write the custom model configuration
# --------------------------------------------------
#
# This is the standard models/yolov5s.yaml with one
# change: nc is 2 (smoke, fire) instead of 80.
# An existing file is never overwritten.
# --------------------------------------------------

CUSTOM_MODEL_CONFIG = r'''# Parameters
nc: 2 # number of classes
depth_multiple: 0.33 # model depth multiple
width_multiple: 0.50 # layer channel multiple
anchors:
  - [10, 13, 16, 30, 33, 23] # P3/8
  - [30, 61, 62, 45, 59, 119] # P4/16
  - [116, 90, 156, 198, 373, 326] # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [
    [-1, 1, Conv, [64, 6, 2, 2]], # 0-P1/2
    [-1, 1, Conv, [128, 3, 2]], # 1-P2/4
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]], # 3-P3/8
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]], # 5-P4/16
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]], # 7-P5/32
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]], # 9
  ]

# YOLOv5 v6.0 head
head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]], # cat backbone P4
    [-1, 3, C3, [512, False]], # 13

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 4], 1, Concat, [1]], # cat backbone P3
    [-1, 3, C3, [256, False]], # 17 (P3/8-small)

    [-1, 1, Conv, [256, 3, 2]],
    [[-1, 14], 1, Concat, [1]], # cat head P4
    [-1, 3, C3, [512, False]], # 20 (P4/16-medium)

    [-1, 1, Conv, [512, 3, 2]],
    [[-1, 10], 1, Concat, [1]], # cat head P5
    [-1, 3, C3, [1024, False]], # 23 (P5/32-large)

    [[17, 20, 23], 1, Detect, [nc, anchors]], # Detect(P3, P4, P5)
  ]
'''

print("\n===== CUSTOM MODEL CONFIG =====")

if CUSTOM_MODEL_YAML.exists():
    print("Already present:", CUSTOM_MODEL_YAML)
else:
    CUSTOM_MODEL_YAML.write_text(CUSTOM_MODEL_CONFIG)
    print("Created:", CUSTOM_MODEL_YAML)


# --------------------------------------------------
# 4. Download the pretrained YOLOv5s weights
# --------------------------------------------------
#
# Training starts from these COCO-pretrained weights.
# --------------------------------------------------

print("\n===== PRETRAINED WEIGHTS =====")

if PRETRAINED_WEIGHTS.exists():
    print("Already present:", PRETRAINED_WEIGHTS)
else:
    urllib.request.urlretrieve(
        WEIGHTS_URL,
        PRETRAINED_WEIGHTS,
    )
    print("Downloaded:", PRETRAINED_WEIGHTS)


# --------------------------------------------------
# 5. Final check
# --------------------------------------------------

print("\n===== SETUP RESULT =====")

required_files = {
    "train.py": YOLOV5_REPO / "train.py",
    "val.py": YOLOV5_REPO / "val.py",
    "custom_yolov5s.yaml": CUSTOM_MODEL_YAML,
    "yolov5s.pt": PRETRAINED_WEIGHTS,
}

for name, path in required_files.items():
    print(name, "->", path.exists())

if all(path.exists() for path in required_files.values()):
    print(
        "\nSUCCESS: YOLOv5, the custom model config "
        "and the pretrained weights are ready."
    )
else:
    raise FileNotFoundError(
        "YOLOv5 setup is incomplete."
    )

## 2. Audit of the original YOLOv5 run

The first YOLOv5 run used a dataset YAML with a train/val/test leakage problem. These cells locate that run and
record its settings so the corrected run can reuse the same hyperparameters. They are read-only.

### 2.1 Locate the existing YOLOv5 repository and results

In [ ]:
from pathlib import Path
import subprocess

# --------------------------------------------------
# 1. Define the project and focused search locations
# --------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/Fire_smoke_detection"
)

search_roots = [
    PROJECT / "external",
    PROJECT / "experiments",
    PROJECT / "configs",
    PROJECT / "results",
    PROJECT / "data" / "yolov5",
]


# --------------------------------------------------
# 2. Display the immediate contents of external/
# --------------------------------------------------
#
# This helps us see the actual folder names and
# capitalisation used in Google Drive.
# --------------------------------------------------

EXTERNAL = PROJECT / "external"

print("===== EXTERNAL FOLDER CONTENTS =====")
print("External folder exists:", EXTERNAL.exists())

if EXTERNAL.exists():
    for item in sorted(EXTERNAL.iterdir()):
        item_type = (
            "FOLDER"
            if item.is_dir()
            else "FILE"
        )

        print(
            item_type,
            "->",
            item,
        )


# --------------------------------------------------
# 3. Locate all train.py files in focused locations
# --------------------------------------------------
#
# A YOLOv5 repository should contain train.py.
# The parent folder of train.py is a repository
# candidate.
# --------------------------------------------------

print("\n===== YOLOv5 train.py FILES =====")

train_scripts = []

for root in search_roots:
    if root.exists():
        train_scripts.extend(
            root.rglob("train.py")
        )

train_scripts = sorted(
    set(train_scripts)
)

repository_candidates = []

if train_scripts:
    for script in train_scripts:
        repository = script.parent
        repository_candidates.append(repository)

        print("\ntrain.py:", script)
        print(
            "Repository candidate:",
            repository,
        )
        print(
            ".git exists:",
            (repository / ".git").exists(),
        )

        if (repository / ".git").exists():
            commit_result = subprocess.run(
                ["git", "rev-parse", "HEAD"],
                cwd=repository,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                check=False,
            )

            print(
                "Git commit:",
                commit_result.stdout.strip(),
            )
else:
    print("No train.py files found.")


# --------------------------------------------------
# 4. Locate previous training-result files
# --------------------------------------------------

result_names = [
    "results.csv",
    "results.txt",
]

print("\n===== YOLOv5 RESULT FILES =====")

result_files = []

for root in search_roots:
    if root.exists():
        for filename in result_names:
            result_files.extend(
                root.rglob(filename)
            )

result_files = sorted(
    set(result_files)
)

if result_files:
    for file in result_files:
        print(file)
else:
    print(
        "No results.csv or results.txt files found."
    )


# --------------------------------------------------
# 5. Locate saved training settings
# --------------------------------------------------

print("\n===== YOLOv5 TRAINING SETTINGS =====")

settings_names = [
    "opt.yaml",
    "args.yaml",
    "hyp.yaml",
]

settings_files = []

for root in search_roots:
    if root.exists():
        for filename in settings_names:
            settings_files.extend(
                root.rglob(filename)
            )

settings_files = sorted(
    set(settings_files)
)

if settings_files:
    for file in settings_files:
        print(file)
else:
    print(
        "No opt.yaml, args.yaml or hyp.yaml "
        "files found."
    )


# --------------------------------------------------
# 6. Locate previous YOLOv5 weights
# --------------------------------------------------

print("\n===== YOLOv5 WEIGHT FILES =====")

weight_names = [
    "best.pt",
    "last.pt",
]

weight_files = []

for root in search_roots:
    if root.exists():
        for filename in weight_names:
            weight_files.extend(
                root.rglob(filename)
            )

weight_files = sorted(
    set(weight_files)
)

if weight_files:
    for file in weight_files:
        size_mb = (
            file.stat().st_size
            / (1024 * 1024)
        )

        print(
            file,
            f"-> {size_mb:.2f} MB",
        )
else:
    print(
        "No best.pt or last.pt files found."
    )


# --------------------------------------------------
# 7. Locate possible dataset YAML files
# --------------------------------------------------
#
# We look only in configs and experiment folders,
# avoiding the hundreds of standard YAML files that
# may exist inside the YOLOv5 repository.
# --------------------------------------------------

print("\n===== DATASET YAML CANDIDATES =====")

yaml_roots = [
    PROJECT / "configs",
    PROJECT / "experiments",
]

dataset_yaml_candidates = []

for root in yaml_roots:
    if root.exists():
        for pattern in [
            "*.yaml",
            "*.yml",
        ]:
            dataset_yaml_candidates.extend(
                root.rglob(pattern)
            )

dataset_yaml_candidates = sorted(
    set(dataset_yaml_candidates)
)

if dataset_yaml_candidates:
    for file in dataset_yaml_candidates:
        print(file)
else:
    print(
        "No project dataset YAML files found."
    )


# --------------------------------------------------
# 8. Print the search conclusion
# --------------------------------------------------

print("\n===== YOLOv5 SEARCH CONCLUSION =====")
print(
    "Repository candidates:",
    len(set(repository_candidates)),
)
print(
    "Result files:",
    len(result_files),
)
print(
    "Settings files:",
    len(settings_files),
)
print(
    "Weight files:",
    len(weight_files),
)
print(
    "Dataset YAML candidates:",
    len(dataset_yaml_candidates),
)

if repository_candidates:
    print(
        "A YOLOv5 repository was found."
    )
else:
    print(
        "No persistent YOLOv5 repository was found. "
        "A new temporary clone may be required."
    )

===== EXTERNAL FOLDER CONTENTS =====
External folder exists: True
FOLDER -> /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv4_darknet
FOLDER -> /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5

===== YOLOv5 train.py FILES =====

train.py: /content/drive/MyDrive/Fire_smoke_detection/data/yolov5/classify/train.py
Repository candidate: /content/drive/MyDrive/Fire_smoke_detection/data/yolov5/classify
.git exists: False

train.py: /content/drive/MyDrive/Fire_smoke_detection/data/yolov5/segment/train.py
Repository candidate: /content/drive/MyDrive/Fire_smoke_detection/data/yolov5/segment
.git exists: False

train.py: /content/drive/MyDrive/Fire_smoke_detection/data/yolov5/train.py
Repository candidate: /content/drive/MyDrive/Fire_smoke_detection/data/yolov5
.git exists: True
Git commit: 0675a31b96ad01746f8f29fe7f08d6a05200b370

train.py: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5/classify/train.py
Repository candidate: /content/drive/MyDrive/Fire_smoke_d

### 2.2 Inspect the original repository, dataset YAML and model config

In [ ]:
from pathlib import Path
import subprocess

# --------------------------------------------------
# 1. Define the exact paths shown in the screenshots
# --------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/Fire_smoke_detection"
)

YOLOV5_REPO = (
    PROJECT
    / "external"
    / "YOLOv5"
)

TRAIN_SCRIPT = YOLOV5_REPO / "train.py"

DATA_YAML = (
    PROJECT
    / "data"
    / "DFire.yaml"
)

CUSTOM_MODEL_YAML = (
    YOLOV5_REPO
    / "models"
    / "custom_yolov5s.yaml"
)


# --------------------------------------------------
# 2. Verify the repository
# --------------------------------------------------

print("===== ORIGINAL YOLOv5 REPOSITORY =====")
print("Repository:", YOLOV5_REPO)
print(
    "Repository exists:",
    YOLOV5_REPO.exists(),
)
print(
    "train.py exists:",
    TRAIN_SCRIPT.exists(),
)
print(
    ".git exists:",
    (YOLOV5_REPO / ".git").exists(),
)

if (
    YOLOV5_REPO.exists()
    and (YOLOV5_REPO / ".git").exists()
):
    commit_result = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=YOLOV5_REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    print(
        "Git commit:",
        commit_result.stdout.strip(),
    )


# --------------------------------------------------
# 3. Display the original dataset YAML
# --------------------------------------------------
#
# This tells us which train, validation and test
# folders the old run used.
# --------------------------------------------------

print("\n===== ORIGINAL DFire.yaml =====")
print("File:", DATA_YAML)
print("Exists:", DATA_YAML.exists())

if DATA_YAML.exists():
    print("\nContents:")
    print(DATA_YAML.read_text())
else:
    print(
        "The original DFire.yaml was not found "
        "at the screenshot path."
    )


# --------------------------------------------------
# 4. Inspect the custom YOLOv5s model configuration
# --------------------------------------------------
#
# Printing the whole model file would be lengthy.
# We print the settings that identify the model size,
# number of classes and anchors.
# --------------------------------------------------

print("\n===== CUSTOM YOLOv5s MODEL CONFIG =====")
print("File:", CUSTOM_MODEL_YAML)
print(
    "Exists:",
    CUSTOM_MODEL_YAML.exists(),
)

if CUSTOM_MODEL_YAML.exists():
    model_lines = (
        CUSTOM_MODEL_YAML
        .read_text()
        .splitlines()
    )

    keys_to_show = [
        "nc:",
        "depth_multiple:",
        "width_multiple:",
        "anchors:",
    ]

    for line_number, line in enumerate(
        model_lines,
        start=1,
    ):
        stripped = line.strip()

        if any(
            stripped.startswith(key)
            for key in keys_to_show
        ):
            print(
                f"Line {line_number}: {line}"
            )

    print(
        "Total configuration lines:",
        len(model_lines),
    )


# --------------------------------------------------
# 5. Locate runs named yolov5_results
# --------------------------------------------------
#
# The --name yolov5_results argument normally creates
# a folder with this name, possibly followed by a
# number if the name already existed.
# --------------------------------------------------

print("\n===== ORIGINAL yolov5_results RUNS =====")

focused_roots = [
    YOLOV5_REPO / "runs",
    PROJECT / "experiments",
    PROJECT / "results",
]

run_directories = []

for root in focused_roots:
    if root.exists():
        for path in root.rglob("*"):
            if (
                path.is_dir()
                and path.name.startswith(
                    "yolov5_results"
                )
            ):
                run_directories.append(path)

run_directories = sorted(
    set(run_directories)
)

if run_directories:
    for run in run_directories:
        print(f"\nRun folder: {run}")

        expected_artifacts = [
            run / "results.csv",
            run / "results.txt",
            run / "opt.yaml",
            run / "args.yaml",
            run / "hyp.yaml",
            run / "weights" / "best.pt",
            run / "weights" / "last.pt",
        ]

        for artifact in expected_artifacts:
            if artifact.exists():
                size_mb = (
                    artifact.stat().st_size
                    / (1024 * 1024)
                )

                print(
                    artifact.name,
                    "-> True",
                    f"({size_mb:.2f} MB)",
                )
else:
    print(
        "No run folder beginning with "
        "yolov5_results was found."
    )


# --------------------------------------------------
# 6. Search for exact run-name references
# --------------------------------------------------
#
# This catches runs stored in a different project
# directory while avoiding the raw dataset.
# --------------------------------------------------

print("\n===== RELATED YOLOv5 ARTIFACTS =====")

artifact_roots = [
    PROJECT / "external",
    PROJECT / "experiments",
    PROJECT / "results",
]

related_files = []

for root in artifact_roots:
    if root.exists():
        for filename in [
            "results.csv",
            "results.txt",
            "opt.yaml",
            "args.yaml",
            "best.pt",
            "last.pt",
        ]:
            for file in root.rglob(filename):
                if "yolov5_results" in str(file):
                    related_files.append(file)

related_files = sorted(
    set(related_files)
)

if related_files:
    for file in related_files:
        print(file)
else:
    print(
        "No files containing yolov5_results "
        "in their path were found."
    )


# --------------------------------------------------
# 7. Print the targeted audit result
# --------------------------------------------------

print("\n===== TARGETED AUDIT CONCLUSION =====")
print(
    "Repository usable:",
    TRAIN_SCRIPT.exists(),
)
print(
    "Original DFire.yaml available:",
    DATA_YAML.exists(),
)
print(
    "Custom model YAML available:",
    CUSTOM_MODEL_YAML.exists(),
)
print(
    "Original run folders found:",
    len(run_directories),
)

if (
    TRAIN_SCRIPT.exists()
    and DATA_YAML.exists()
    and CUSTOM_MODEL_YAML.exists()
):
    print(
        "SUCCESS: The original YOLOv5 method "
        "can be reproduced for corrected training."
    )
else:
    print(
        "Some original components are missing. "
        "We will recreate only the missing items."
    )

===== ORIGINAL YOLOv5 REPOSITORY =====
Repository: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5
Repository exists: True
train.py exists: True
.git exists: False

===== ORIGINAL DFire.yaml =====
File: /content/drive/MyDrive/Fire_smoke_detection/data/DFire.yaml
Exists: True

Contents:
path: /content/drive/MyDrive/Fire_smoke_detection/data/raw/DFireDataset

train: train/images
val: test/images
test: test/images

nc: 2

names:
  0: smoke
  1: fire

===== CUSTOM YOLOv5s MODEL CONFIG =====
File: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5/models/custom_yolov5s.yaml
Exists: True
Line 3: nc: 2 # number of classes
Line 4: depth_multiple: 0.33 # model depth multiple
Line 5: width_multiple: 0.50 # layer channel multiple
Line 6: anchors:
Total configuration lines: 48

===== ORIGINAL yolov5_results RUNS =====

Run folder: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5/runs/train/yolov5_results
opt.yaml -> True (0.00 MB)
hyp.yaml -> True (0.00 MB)

Run fol

### 2.3 Identify the completed original run

In [ ]:
from pathlib import Path
from datetime import datetime
import yaml
import pandas as pd

# --------------------------------------------------
# 1. Define the original YOLOv5 paths
# --------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/Fire_smoke_detection"
)

YOLOV5_REPO = (
    PROJECT
    / "external"
    / "YOLOv5"
)

DATA_YAML = (
    PROJECT
    / "data"
    / "DFire.yaml"
)

RUNS_ROOT = (
    YOLOV5_REPO
    / "runs"
    / "train"
)


# --------------------------------------------------
# 2. Display the complete original dataset YAML
# --------------------------------------------------
#
# This confirms which image folders were used by the
# previous YOLOv5 experiment.
# --------------------------------------------------

print("===== ORIGINAL DFire.yaml =====")
print("File:", DATA_YAML)
print("Exists:", DATA_YAML.exists())

if not DATA_YAML.exists():
    raise FileNotFoundError(
        "The original DFire.yaml was not found."
    )

print("\nContents:")
print(DATA_YAML.read_text())

try:
    data_yaml_values = yaml.safe_load(
        DATA_YAML.read_text()
    )

    print("Parsed YAML:")
    print(data_yaml_values)
except Exception as error:
    print(
        "YAML parsing error:",
        error,
    )


# --------------------------------------------------
# 3. Locate every yolov5_results run directory
# --------------------------------------------------

print("\n===== ORIGINAL RUN DIRECTORY CHECK =====")
print("Runs root:", RUNS_ROOT)
print("Runs root exists:", RUNS_ROOT.exists())

if not RUNS_ROOT.exists():
    raise FileNotFoundError(
        "The YOLOv5 runs/train folder is missing."
    )

run_directories = sorted(
    [
        folder
        for folder in RUNS_ROOT.iterdir()
        if (
            folder.is_dir()
            and folder.name.startswith(
                "yolov5_results"
            )
        )
    ],
    key=lambda folder: folder.stat().st_mtime,
)

print(
    "Run directories found:",
    len(run_directories),
)


# --------------------------------------------------
# 4. Inspect the artifacts and settings of each run
# --------------------------------------------------

run_summaries = []

for run in run_directories:
    results_csv = run / "results.csv"
    results_txt = run / "results.txt"
    opt_yaml = run / "opt.yaml"
    args_yaml = run / "args.yaml"

    best_pt = run / "weights" / "best.pt"
    last_pt = run / "weights" / "last.pt"

    recorded_epochs = 0
    results_type = "none"

    # Newer/compatible YOLOv5 result format
    if results_csv.exists():
        try:
            results_df = pd.read_csv(
                results_csv
            )

            recorded_epochs = len(results_df)
            results_type = "results.csv"
        except Exception:
            results_type = (
                "results.csv unreadable"
            )

    # Some older YOLOv5 versions use results.txt
    elif results_txt.exists():
        result_lines = [
            line
            for line in results_txt.read_text(
                errors="replace"
            ).splitlines()
            if line.strip()
        ]

        recorded_epochs = len(result_lines)
        results_type = "results.txt"

    settings_file = None
    settings = {}

    if opt_yaml.exists():
        settings_file = opt_yaml
    elif args_yaml.exists():
        settings_file = args_yaml

    if settings_file:
        try:
            settings = yaml.safe_load(
                settings_file.read_text()
            ) or {}
        except Exception:
            settings = {}

    configured_epochs = settings.get(
        "epochs"
    )

    summary = {
        "run": run.name,
        "modified": datetime.fromtimestamp(
            run.stat().st_mtime
        ).isoformat(
            timespec="seconds"
        ),
        "result_format": results_type,
        "recorded_epochs": recorded_epochs,
        "configured_epochs": configured_epochs,
        "img_size": settings.get(
            "img_size",
            settings.get("imgsz"),
        ),
        "batch_size": settings.get(
            "batch_size",
            settings.get("batch"),
        ),
        "weights_setting": settings.get(
            "weights"
        ),
        "cfg_setting": settings.get("cfg"),
        "data_setting": settings.get("data"),
        "hyp_setting": settings.get("hyp"),
        "resume": settings.get("resume"),
        "best_pt": best_pt.exists(),
        "last_pt": last_pt.exists(),
        "best_size_mb": (
            best_pt.stat().st_size
            / (1024 * 1024)
            if best_pt.exists()
            else 0
        ),
    }

    run_summaries.append(summary)


# --------------------------------------------------
# 5. Print a compact summary for every run
# --------------------------------------------------

print("\n===== RUN SUMMARIES =====")

summary_df = pd.DataFrame(
    run_summaries
)

display_columns = [
    "run",
    "modified",
    "result_format",
    "recorded_epochs",
    "configured_epochs",
    "img_size",
    "batch_size",
    "best_pt",
    "last_pt",
    "resume",
]

print(
    summary_df[display_columns]
    .to_string(index=False)
)


# --------------------------------------------------
# 6. Identify likely completed 85-epoch runs
# --------------------------------------------------

completed_runs = summary_df[
    (
        summary_df["recorded_epochs"]
        >= 85
    )
    & (
        summary_df["best_pt"]
        == True
    )
]

print("\n===== LIKELY COMPLETED 85-EPOCH RUNS =====")

if not completed_runs.empty:
    print(
        completed_runs[
            [
                "run",
                "recorded_epochs",
                "img_size",
                "batch_size",
                "weights_setting",
                "cfg_setting",
                "data_setting",
                "hyp_setting",
                "resume",
                "best_size_mb",
            ]
        ].to_string(index=False)
    )
else:
    print(
        "No run with at least 85 recorded epochs "
        "and best.pt was found."
    )


# --------------------------------------------------
# 7. Print the most complete candidate's settings
# --------------------------------------------------

if not completed_runs.empty:
    selected_name = completed_runs.iloc[-1][
        "run"
    ]

    selected_summary = next(
        item
        for item in run_summaries
        if item["run"] == selected_name
    )

    selected_run = (
        RUNS_ROOT / selected_name
    )

    selected_opt = selected_run / "opt.yaml"
    selected_args = selected_run / "args.yaml"

    selected_settings_file = (
        selected_opt
        if selected_opt.exists()
        else selected_args
    )

    print(
        "\n===== MOST RECENT COMPLETE "
        "RUN SETTINGS ====="
    )

    print("Selected run:", selected_run)
    print(
        "Settings file:",
        selected_settings_file,
    )

    if selected_settings_file.exists():
        print(
            selected_settings_file.read_text()
        )


# --------------------------------------------------
# 8. Final conclusion
# --------------------------------------------------

print("\n===== COMPLETION AUDIT RESULT =====")
print(
    "Completed 85-epoch candidates:",
    len(completed_runs),
)

if not completed_runs.empty:
    print(
        "SUCCESS: At least one complete original "
        "YOLOv5 run was identified."
    )
else:
    print(
        "The screenshots will be used to recreate "
        "the original command because no complete "
        "run was identified automatically."
    )

===== ORIGINAL DFire.yaml =====
File: /content/drive/MyDrive/Fire_smoke_detection/data/DFire.yaml
Exists: True

Contents:
path: /content/drive/MyDrive/Fire_smoke_detection/data/raw/DFireDataset

train: train/images
val: test/images
test: test/images

nc: 2

names:
  0: smoke
  1: fire
Parsed YAML:
{'path': '/content/drive/MyDrive/Fire_smoke_detection/data/raw/DFireDataset', 'train': 'train/images', 'val': 'test/images', 'test': 'test/images', 'nc': 2, 'names': {0: 'smoke', 1: 'fire'}}

===== ORIGINAL RUN DIRECTORY CHECK =====
Runs root: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5/runs/train
Runs root exists: True
Run directories found: 23

===== RUN SUMMARIES =====
             run            modified result_format  recorded_epochs  configured_epochs  img_size  batch_size  best_pt  last_pt  resume
  yolov5_results 2026-05-21T20:24:56          none                0                  1       640          16    False    False   False
 yolov5_results2 2026-05-23T17:25:35    

### 2.4 Print the essential settings of the original run

In [ ]:
from pathlib import Path
import yaml
import pandas as pd

# --------------------------------------------------
# 1. Define the known YOLOv5 paths
# --------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/Fire_smoke_detection"
)

YOLOV5_REPO = (
    PROJECT
    / "external"
    / "YOLOv5"
)

RUNS_ROOT = (
    YOLOV5_REPO
    / "runs"
    / "train"
)

DATA_YAML = (
    PROJECT
    / "data"
    / "DFire.yaml"
)


# --------------------------------------------------
# 2. Identify completed 85-epoch runs
# --------------------------------------------------

completed_runs = []

for run in sorted(RUNS_ROOT.glob("yolov5_results*")):
    if not run.is_dir():
        continue

    results_csv = run / "results.csv"
    results_txt = run / "results.txt"
    best_pt = run / "weights" / "best.pt"

    recorded_epochs = 0

    if results_csv.exists():
        try:
            recorded_epochs = len(
                pd.read_csv(results_csv)
            )
        except Exception:
            recorded_epochs = 0

    elif results_txt.exists():
        recorded_epochs = len([
            line
            for line in results_txt.read_text(
                errors="replace"
            ).splitlines()
            if line.strip()
        ])

    if (
        recorded_epochs >= 85
        and best_pt.exists()
    ):
        completed_runs.append({
            "folder": run,
            "epochs": recorded_epochs,
        })


print("===== COMPLETED RUN IDENTIFICATION =====")
print(
    "Completed runs found:",
    len(completed_runs),
)

for item in completed_runs:
    print(
        item["folder"],
        "->",
        item["epochs"],
        "recorded epochs",
    )

if len(completed_runs) != 1:
    raise RuntimeError(
        "Expected exactly one completed run. "
        "Review the listed candidates."
    )

SELECTED_RUN = completed_runs[0]["folder"]


# --------------------------------------------------
# 3. Load the completed run settings
# --------------------------------------------------

OPT_YAML = SELECTED_RUN / "opt.yaml"
ARGS_YAML = SELECTED_RUN / "args.yaml"

SETTINGS_FILE = (
    OPT_YAML
    if OPT_YAML.exists()
    else ARGS_YAML
)

if not SETTINGS_FILE.exists():
    raise FileNotFoundError(
        "The completed run has no opt.yaml "
        "or args.yaml file."
    )

settings = yaml.safe_load(
    SETTINGS_FILE.read_text()
) or {}


# --------------------------------------------------
# 4. Print only scientifically relevant settings
# --------------------------------------------------

important_keys = [
    "weights",
    "cfg",
    "data",
    "hyp",
    "epochs",
    "batch_size",
    "batch",
    "img_size",
    "imgsz",
    "rect",
    "resume",
    "nosave",
    "noval",
    "noautoanchor",
    "evolve",
    "cache_images",
    "cache",
    "image_weights",
    "device",
    "multi_scale",
    "single_cls",
    "adam",
    "optimizer",
    "sync_bn",
    "workers",
    "project",
    "name",
    "exist_ok",
    "quad",
    "linear_lr",
    "label_smoothing",
    "patience",
    "freeze",
    "save_period",
    "seed",
]

print("\n===== ESSENTIAL ORIGINAL SETTINGS =====")
print("Selected run:", SELECTED_RUN)
print("Settings file:", SETTINGS_FILE)

for key in important_keys:
    if key in settings:
        print(
            f"{key}: {settings[key]}"
        )


# --------------------------------------------------
# 5. Print the original dataset YAML
# --------------------------------------------------

print("\n===== ORIGINAL DFire.yaml =====")
print("File:", DATA_YAML)
print("Exists:", DATA_YAML.exists())

if not DATA_YAML.exists():
    raise FileNotFoundError(
        "The original DFire.yaml is missing."
    )

print(DATA_YAML.read_text())

data_settings = yaml.safe_load(
    DATA_YAML.read_text()
) or {}


# --------------------------------------------------
# 6. Show how YAML paths are interpreted
# --------------------------------------------------
#
# YOLOv5 dataset YAML files may use:
#
# path: dataset root
# train: relative train path
# val: relative validation path
# test: relative test path
#
# Or train/val/test may each be absolute paths.
# --------------------------------------------------

print("===== PARSED DATASET SETTINGS =====")

for key in [
    "path",
    "train",
    "val",
    "test",
    "nc",
    "names",
]:
    print(
        f"{key}:",
        data_settings.get(key),
    )


# --------------------------------------------------
# 7. Verify the expected corrected dataset folders
# --------------------------------------------------

CORRECTED_DATASET = (
    PROJECT
    / "data"
    / "processed"
    / "DFireDataset_split"
)

expected_folders = {
    "train_images":
        CORRECTED_DATASET / "images" / "train",

    "val_images":
        CORRECTED_DATASET / "images" / "val",

    "test_images":
        CORRECTED_DATASET / "images" / "test",

    "train_labels":
        CORRECTED_DATASET / "labels" / "train",

    "val_labels":
        CORRECTED_DATASET / "labels" / "val",

    "test_labels":
        CORRECTED_DATASET / "labels" / "test",
}

print("\n===== CORRECTED PATH CHECK =====")

for name, folder in expected_folders.items():
    print(
        name,
        "->",
        folder.exists(),
        "->",
        folder,
    )


# --------------------------------------------------
# 8. Verify the completed run artifacts
# --------------------------------------------------

print("\n===== COMPLETED RUN ARTIFACTS =====")

artifacts = [
    SELECTED_RUN / "results.csv",
    SELECTED_RUN / "results.txt",
    SELECTED_RUN / "weights" / "best.pt",
    SELECTED_RUN / "weights" / "last.pt",
]

for artifact in artifacts:
    print(
        artifact.name,
        "->",
        artifact.exists(),
    )

print("\n===== FOCUSED AUDIT RESULT =====")
print(
    "SUCCESS: Essential original settings "
    "and corrected dataset paths were displayed."
)

===== COMPLETED RUN IDENTIFICATION =====
Completed runs found: 1
/content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5/runs/train/yolov5_results23 -> 85 recorded epochs

===== ESSENTIAL ORIGINAL SETTINGS =====
Selected run: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5/runs/train/yolov5_results23
Settings file: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5/runs/train/yolov5_results23/opt.yaml
weights: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5/runs/train/yolov5_results23/weights/last.pt
cfg: 
data: /content/drive/MyDrive/Fire_smoke_detection/data/DFire.yaml
hyp: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'warmup_bias_lr': 0.1, 'box': 0.05, 'cls': 0.5, 'cls_pw': 1.0, 'obj': 1.0, 'obj_pw': 1.0, 'iou_t': 0.2, 'anchor_t': 4.0, 'fl_gamma': 0.0, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 

## 3. Corrected dataset configuration

Writes `configs/YOLOv5/DFire_corrected.yaml`, pointing to the corrected split. Class order: `0 = smoke`, `1 = fire`.

In [ ]:
from pathlib import Path
import yaml

# --------------------------------------------------
# 1. Define the project and corrected dataset paths
# --------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/Fire_smoke_detection"
)

CORRECTED_DATASET = (
    PROJECT
    / "data"
    / "processed"
    / "DFireDataset_split"
)

YOLOV5_CONFIG_DIR = (
    PROJECT
    / "configs"
    / "YOLOv5"
)

CORRECTED_YAML = (
    YOLOV5_CONFIG_DIR
    / "DFire_corrected.yaml"
)


# --------------------------------------------------
# 2. Verify the corrected dataset before creating
#    the YAML file
# --------------------------------------------------

required_folders = {
    "train images":
        CORRECTED_DATASET / "images" / "train",

    "validation images":
        CORRECTED_DATASET / "images" / "val",

    "test images":
        CORRECTED_DATASET / "images" / "test",

    "train labels":
        CORRECTED_DATASET / "labels" / "train",

    "validation labels":
        CORRECTED_DATASET / "labels" / "val",

    "test labels":
        CORRECTED_DATASET / "labels" / "test",
}

print("===== CORRECTED DATASET FOLDER CHECK =====")

for name, folder in required_folders.items():
    print(
        name,
        "->",
        folder.exists(),
        "->",
        folder,
    )

if not all(
    folder.exists()
    for folder in required_folders.values()
):
    raise FileNotFoundError(
        "One or more corrected dataset folders "
        "are missing. The YAML file was not created."
    )


# --------------------------------------------------
# 3. Create the dedicated YOLOv5 config directory
# --------------------------------------------------
#
# This creates only a small configuration folder.
# It does not copy any images or labels.
# --------------------------------------------------

YOLOV5_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# 4. Define the corrected dataset configuration
# --------------------------------------------------
#
# path:
#     The common dataset root.
#
# train, val and test:
#     Paths relative to the dataset root.
#
# names:
#     The list position defines the class ID:
#     position 0 = smoke
#     position 1 = fire
# --------------------------------------------------

corrected_configuration = {
    "path": str(CORRECTED_DATASET),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 2,
    "names": [
        "smoke",
        "fire",
    ],
}


# --------------------------------------------------
# 5. Write the YAML file
# --------------------------------------------------
#
# sort_keys=False preserves the logical order:
# path, train, val, test, nc, names.
# --------------------------------------------------

with CORRECTED_YAML.open("w") as file:
    yaml.safe_dump(
        corrected_configuration,
        file,
        sort_keys=False,
    )


# --------------------------------------------------
# 6. Read the YAML back and verify its contents
# --------------------------------------------------

loaded_configuration = yaml.safe_load(
    CORRECTED_YAML.read_text()
)

print("\n===== CREATED DFire_corrected.yaml =====")
print(CORRECTED_YAML.read_text())


# --------------------------------------------------
# 7. Resolve the YAML paths exactly as YOLOv5 will
# --------------------------------------------------

dataset_root = Path(
    loaded_configuration["path"]
)

resolved_paths = {
    "train":
        dataset_root
        / loaded_configuration["train"],

    "val":
        dataset_root
        / loaded_configuration["val"],

    "test":
        dataset_root
        / loaded_configuration["test"],
}

print("===== RESOLVED IMAGE PATHS =====")

for split, folder in resolved_paths.items():
    print(
        split,
        "->",
        folder,
        "-> exists:",
        folder.exists(),
    )


# --------------------------------------------------
# 8. Count images and labels for final verification
# --------------------------------------------------

expected_counts = {
    "train": 13777,
    "val": 3444,
    "test": 4306,
}

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
}

all_counts_correct = True

print("\n===== YAML DATASET COUNT CHECK =====")

for split, expected in expected_counts.items():
    image_folder = resolved_paths[split]
    label_folder = (
        dataset_root
        / "labels"
        / split
    )

    images = [
        file
        for file in image_folder.iterdir()
        if (
            file.is_file()
            and file.suffix.lower()
            in image_extensions
        )
    ]

    labels = list(
        label_folder.glob("*.txt")
    )

    count_correct = (
        len(images) == expected
        and len(labels) == expected
    )

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Expected:", expected)
    print("Counts correct:", count_correct)

    if not count_correct:
        all_counts_correct = False


# --------------------------------------------------
# 9. Final verification
# --------------------------------------------------

print("\n===== CORRECTED YAML RESULT =====")
print(
    "YAML file exists:",
    CORRECTED_YAML.exists(),
)
print(
    "Number of classes:",
    loaded_configuration.get("nc"),
)
print(
    "Class names:",
    loaded_configuration.get("names"),
)

if (
    CORRECTED_YAML.exists()
    and all_counts_correct
    and loaded_configuration["nc"] == 2
    and loaded_configuration["names"]
        == ["smoke", "fire"]
):
    print(
        "\nSUCCESS: DFire_corrected.yaml is "
        "ready for fair YOLOv5 training."
    )
else:
    print(
        "\nSTOP: The corrected dataset YAML "
        "did not pass verification."
    )

===== CORRECTED DATASET FOLDER CHECK =====
train images -> True -> /content/drive/MyDrive/Fire_smoke_detection/data/processed/DFireDataset_split/images/train
validation images -> True -> /content/drive/MyDrive/Fire_smoke_detection/data/processed/DFireDataset_split/images/val
test images -> True -> /content/drive/MyDrive/Fire_smoke_detection/data/processed/DFireDataset_split/images/test
train labels -> True -> /content/drive/MyDrive/Fire_smoke_detection/data/processed/DFireDataset_split/labels/train
validation labels -> True -> /content/drive/MyDrive/Fire_smoke_detection/data/processed/DFireDataset_split/labels/val
test labels -> True -> /content/drive/MyDrive/Fire_smoke_detection/data/processed/DFireDataset_split/labels/test

===== CREATED DFire_corrected.yaml =====
path: /content/drive/MyDrive/Fire_smoke_detection/data/processed/DFireDataset_split
train: images/train
val: images/val
test: images/test
nc: 2
names:
- smoke
- fire

===== RESOLVED IMAGE PATHS =====
train -> /content/drive

## 4. Install YOLOv5 dependencies

Installs the repository requirements, then checks the Ultralytics import, GPU access and that `train.py` loads.

In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

# --------------------------------------------------
# 1. Define the YOLOv5 repository files
# --------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/Fire_smoke_detection"
)

YOLOV5_REPO = (
    PROJECT
    / "external"
    / "YOLOv5"
)

TRAIN_SCRIPT = (
    YOLOV5_REPO
    / "train.py"
)

REQUIREMENTS = (
    YOLOV5_REPO
    / "requirements.txt"
)


# --------------------------------------------------
# 2. Verify the dependency file
# --------------------------------------------------

print("===== DEPENDENCY FILE CHECK =====")
print(
    "requirements.txt exists:",
    REQUIREMENTS.exists(),
)
print(
    "train.py exists:",
    TRAIN_SCRIPT.exists(),
)

if not (
    REQUIREMENTS.exists()
    and TRAIN_SCRIPT.exists()
):
    raise FileNotFoundError(
        "The YOLOv5 requirements or training "
        "script is missing."
    )


# --------------------------------------------------
# 3. Display relevant requirements
# --------------------------------------------------
#
# We show lines related to PyTorch and Ultralytics
# so it is clear what the repository expects.
# --------------------------------------------------

requirements_lines = (
    REQUIREMENTS.read_text()
    .splitlines()
)

print("\n===== RELEVANT REQUIREMENT LINES =====")

relevant_lines = [
    line
    for line in requirements_lines
    if (
        "ultralytics" in line.lower()
        or "torch" in line.lower()
    )
]

if relevant_lines:
    for line in relevant_lines:
        print(line)
else:
    print(
        "No explicit ultralytics or torch "
        "requirement line was found."
    )


# --------------------------------------------------
# 4. Install the repository requirements
# --------------------------------------------------
#
# sys.executable ensures pip installs packages into
# the same Python environment used by this notebook.
# --------------------------------------------------

print("\n===== INSTALLING YOLOv5 DEPENDENCIES =====")
print(
    "This may take several minutes."
)

install_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(REQUIREMENTS),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

install_lines = (
    install_result.stdout.splitlines()
)

print("\n===== FINAL 80 INSTALLATION LINES =====")

for line in install_lines[-80:]:
    print(line)

print(
    "\npip exit code:",
    install_result.returncode,
)

if install_result.returncode != 0:
    raise RuntimeError(
        "Dependency installation failed. "
        "Inspect the installation output above."
    )


# --------------------------------------------------
# 5. Verify the Ultralytics package
# --------------------------------------------------

print("\n===== ULTRALYTICS IMPORT CHECK =====")

try:
    import ultralytics

    print(
        "Ultralytics version:",
        ultralytics.__version__,
    )
    print(
        "Ultralytics import: SUCCESS"
    )

except Exception as error:
    print(
        "Ultralytics import error:",
        repr(error),
    )

    raise


# --------------------------------------------------
# 6. Confirm PyTorch GPU access remains available
# --------------------------------------------------

print("\n===== PYTORCH GPU RECHECK =====")
print("PyTorch version:", torch.__version__)
print(
    "CUDA available:",
    torch.cuda.is_available(),
)

if torch.cuda.is_available():
    print(
        "CUDA device:",
        torch.cuda.get_device_name(0),
    )
else:
    raise RuntimeError(
        "PyTorch lost access to the GPU "
        "after dependency installation."
    )


# --------------------------------------------------
# 7. Retest train.py without starting training
# --------------------------------------------------
#
# --help loads the training modules and arguments.
# It does not train or create a run folder.
# --------------------------------------------------

print("\n===== train.py LOADING RETEST =====")

help_result = subprocess.run(
    [
        sys.executable,
        str(TRAIN_SCRIPT),
        "--help",
    ],
    cwd=YOLOV5_REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

print(
    "train.py --help exit code:",
    help_result.returncode,
)

if help_result.returncode != 0:
    print("\n===== REMAINING ERROR OUTPUT =====")
    print(help_result.stdout)

    raise RuntimeError(
        "train.py still cannot load."
    )

print(
    "SUCCESS: YOLOv5 dependencies are installed "
    "and train.py loads correctly."
)

[Output condensed for readability: progress bars and repeated warnings removed. The full console log is saved under experiments/YOLOv5/logs/]

===== DEPENDENCY FILE CHECK =====
requirements.txt exists: True
train.py exists: True

===== RELEVANT REQUIREMENT LINES =====
torch>=1.8.0  # see https://pytorch.org/get-started/locally (recommended)
torchvision>=0.9.0
ultralytics>=8.4.47  # https://ultralytics.com
# protobuf<=3.20.1  # https://github.com/ultralytics/yolov5/issues/8012

===== INSTALLING YOLOv5 DEPENDENCIES =====
This may take several minutes.

===== FINAL 80 INSTALLATION LINES =====
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0

pip exit

## 5. Training

### 5.1 Start the corrected training run

Launches `train.py` for 85 epochs and writes every console line to a log on Drive. The cell refuses to start
if a training process is active or if the run folder already contains progress; in that case use 5.2.

The saved output below is from the first Colab session, which disconnected during epoch 27.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

# --------------------------------------------------
# 1. Define all paths
# --------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/Fire_smoke_detection"
)

YOLOV5_REPO = (
    PROJECT
    / "external"
    / "YOLOv5"
)

TRAIN_SCRIPT = (
    YOLOV5_REPO
    / "train.py"
)

CORRECTED_DATA_YAML = (
    PROJECT
    / "configs"
    / "YOLOv5"
    / "DFire_corrected.yaml"
)

CUSTOM_MODEL_YAML = (
    YOLOV5_REPO
    / "models"
    / "custom_yolov5s.yaml"
)

PRETRAINED_WEIGHTS = (
    YOLOV5_REPO
    / "yolov5s.pt"
)

EXPERIMENT_PROJECT = (
    PROJECT
    / "experiments"
    / "YOLOv5"
)

RUN_NAME = (
    "yolov5s_corrected_416_85"
)

RUN_DIRECTORY = (
    EXPERIMENT_PROJECT
    / RUN_NAME
)

LOG_DIRECTORY = (
    EXPERIMENT_PROJECT
    / "logs"
)


# --------------------------------------------------
# 2. Verify the required input files
# --------------------------------------------------

required_files = [
    TRAIN_SCRIPT,
    CORRECTED_DATA_YAML,
    CUSTOM_MODEL_YAML,
    PRETRAINED_WEIGHTS,
]

print("===== CONSOLIDATED INPUT CHECK =====")

for file in required_files:
    print(file, "->", file.exists())

if not all(file.exists() for file in required_files):
    missing = [
        str(file)
        for file in required_files
        if not file.exists()
    ]

    raise FileNotFoundError(
        "Missing input files:\n"
        + "\n".join(missing)
    )


# --------------------------------------------------
# 3. Confirm that no train.py process is active
# --------------------------------------------------

print("\n===== ACTIVE PROCESS CHECK =====")

process_check = subprocess.run(
    [
        "pgrep",
        "-af",
        "train.py",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

process_output = (
    process_check.stdout.strip()
)

print(
    "pgrep exit code:",
    process_check.returncode,
)

if process_output:
    print(process_output)
else:
    print(
        "No active train.py process was found."
    )

active_training = (
    process_check.returncode == 0
    and "python" in process_output.lower()
    and "train.py" in process_output.lower()
)

if active_training:
    raise RuntimeError(
        "A train.py process is active. "
        "Nothing was renamed or started."
    )


# --------------------------------------------------
# 4. Preserve the current incomplete run if present
# --------------------------------------------------
#
# A folder is considered safely incomplete only when
# it has neither last.pt nor results.csv.
# --------------------------------------------------

print("\n===== EXISTING RUN HANDLING =====")

if RUN_DIRECTORY.exists():
    existing_last = (
        RUN_DIRECTORY
        / "weights"
        / "last.pt"
    )

    existing_results = (
        RUN_DIRECTORY
        / "results.csv"
    )

    print(
        "Existing run folder:",
        RUN_DIRECTORY,
    )
    print(
        "Existing last.pt:",
        existing_last.exists(),
    )
    print(
        "Existing results.csv:",
        existing_results.exists(),
    )

    if (
        existing_last.exists()
        or existing_results.exists()
    ):
        raise RuntimeError(
            "The existing run now contains progress. "
            "It was not renamed. Use the resume "
            "procedure instead."
        )

    backup_timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    incomplete_backup = (
        EXPERIMENT_PROJECT
        / (
            RUN_NAME
            + "_incomplete_"
            + backup_timestamp
        )
    )

    RUN_DIRECTORY.rename(
        incomplete_backup
    )

    print(
        "Preserved incomplete run as:",
        incomplete_backup,
    )

else:
    print(
        "No existing corrected run folder "
        "needed preservation."
    )

print(
    "Intended run name now available:",
    not RUN_DIRECTORY.exists(),
)

if RUN_DIRECTORY.exists():
    raise RuntimeError(
        "The intended run name is still occupied."
    )


# --------------------------------------------------
# 5. Create a unique permanent console log
# --------------------------------------------------

EXPERIMENT_PROJECT.mkdir(
    parents=True,
    exist_ok=True,
)

LOG_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

launch_timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

CONSOLE_LOG = (
    LOG_DIRECTORY
    / (
        RUN_NAME
        + "_console_"
        + launch_timestamp
        + ".log"
    )
)


# --------------------------------------------------
# 6. Verify train.py can load
# --------------------------------------------------

print("\n===== train.py LOADING CHECK =====")

help_result = subprocess.run(
    [
        sys.executable,
        str(TRAIN_SCRIPT),
        "--help",
    ],
    cwd=YOLOV5_REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

print(
    "train.py --help exit code:",
    help_result.returncode,
)

if help_result.returncode != 0:
    print(help_result.stdout)

    raise RuntimeError(
        "train.py cannot load. "
        "Training was not started."
    )


# --------------------------------------------------
# 7. Construct the stable training command
# --------------------------------------------------
#
# --cache is intentionally omitted.
#
# The other settings match the original experiment:
# image size 416
# batch size 16
# 85 epochs
# YOLOv5s
# pretrained yolov5s.pt
# --------------------------------------------------

command = [
    sys.executable,
    "-u",
    str(TRAIN_SCRIPT),

    "--img", "416",
    "--batch", "16",
    "--epochs", "85",

    "--data",
    str(CORRECTED_DATA_YAML),

    "--cfg",
    str(CUSTOM_MODEL_YAML),

    "--weights",
    str(PRETRAINED_WEIGHTS),

    "--project",
    str(EXPERIMENT_PROJECT),

    "--name",
    RUN_NAME,
]


# --------------------------------------------------
# 8. Create and verify the console log
# --------------------------------------------------

start_time = datetime.now().isoformat()

with CONSOLE_LOG.open("w") as log:
    log.write(
        "Corrected YOLOv5s stable training\n"
        f"Started: {start_time}\n"
        "RAM caching: disabled\n"
        f"Command: {' '.join(command)}\n\n"
    )
    log.flush()

print("\n===== LOGGING CHECK =====")
print(
    "Console log:",
    CONSOLE_LOG,
)
print(
    "Console log exists:",
    CONSOLE_LOG.exists(),
)

if not CONSOLE_LOG.exists():
    raise RuntimeError(
        "The console log was not created."
    )


# --------------------------------------------------
# 9. Display the final settings
# --------------------------------------------------

print("\n===== STARTING STABLE CORRECTED RUN =====")
print("Image size: 416")
print("Batch size: 16")
print("Epochs: 85")
print("Seed: repository default 0")
print("Multi-scale: disabled")
print("RAM caching: disabled")
print("Resume: False")
print("Run directory:", RUN_DIRECTORY)
print("\nTraining output begins below:\n")


# --------------------------------------------------
# 10. Launch training and record every output line
# --------------------------------------------------

with CONSOLE_LOG.open("a") as log:
    process = subprocess.Popen(
        command,
        cwd=YOLOV5_REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

    exit_code = process.wait()


# --------------------------------------------------
# 11. Report the final process state
# --------------------------------------------------

LAST_WEIGHTS = (
    RUN_DIRECTORY
    / "weights"
    / "last.pt"
)

RESULTS_CSV = (
    RUN_DIRECTORY
    / "results.csv"
)

print("\n===== PROCESS ENDED =====")
print("Exit code:", exit_code)
print(
    "last.pt exists:",
    LAST_WEIGHTS.exists(),
)
print(
    "results.csv exists:",
    RESULTS_CSV.exists(),
)
print("Console log:", CONSOLE_LOG)

if exit_code == 0:
    print(
        "SUCCESS: All 85 epochs completed."
    )

elif LAST_WEIGHTS.exists():
    print(
        "Training stopped after saving progress. "
        "The run is resumable."
    )

else:
    print(
        "Training failed before a checkpoint. "
        "The console log contains the cause."
    )

[Output condensed for readability: progress bars and repeated warnings removed. The full console log is saved under experiments/YOLOv5/logs/]

Streaming output truncated to the last 5000 lines.

                   all       3444       4245      0.755      0.673      0.723      0.386

      Epoch    GPU_mem   box_loss   obj_loss   cls_loss  Instances       Size

                   all       3444       4245      0.773      0.655      0.727      0.388

      Epoch    GPU_mem   box_loss   obj_loss   cls_loss  Instances       Size


### 5.2 Resume training from `last.pt`

Used after the Colab disconnect. The saved output runs through the final epoch.

In [ ]:
from pathlib import Path
import subprocess
from datetime import datetime

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

YOLOV5_DIR = PROJECT / "external" / "YOLOv5"
RUN_DIR = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "yolov5s_corrected_416_85"
)

TRAIN_SCRIPT = YOLOV5_DIR / "train.py"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"
LOG_DIR = PROJECT / "experiments" / "YOLOv5" / "logs"

print("===== RESUME PRECHECK =====")

required_files = {
    "train.py": TRAIN_SCRIPT,
    "last.pt": LAST_WEIGHTS,
    "results.csv": RESULTS_CSV,
}

all_available = True

for name, path in required_files.items():
    exists = path.exists()
    print(f"{name}: {path} -> {exists}")

    if not exists:
        all_available = False

# Prevent two training processes from using the GPU simultaneously
process_check = subprocess.run(
    ["pgrep", "-af", "train.py"],
    text=True,
    capture_output=True
)

print("\n===== ACTIVE PROCESS CHECK =====")

if process_check.returncode == 0:
    print(process_check.stdout.strip())
    raise RuntimeError(
        "Another train.py process is active. "
        "Resume was not started."
    )
else:
    print("No active train.py process was found.")

if not all_available:
    raise FileNotFoundError(
        "One or more required resume files are missing. "
        "Resume was not started."
    )

# Create a separate log for this resume session
LOG_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RESUME_LOG = (
    LOG_DIR
    / f"yolov5s_corrected_416_85_resume_{timestamp}.log"
)

command = [
    "python",
    "-u",
    str(TRAIN_SCRIPT),
    "--resume",
    str(LAST_WEIGHTS),
]

print("\n===== RESUME COMMAND =====")
print(" ".join(command))

print("\nResume log:")
print(RESUME_LOG)

print("\n===== RESUMING TRAINING =====")
print("Training output will appear below.\n")

with RESUME_LOG.open(
    "w",
    encoding="utf-8",
    buffering=1
) as log_file:

    process = subprocess.Popen(
        command,
        cwd=YOLOV5_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

    exit_code = process.wait()

print("\n===== RESUME SESSION ENDED =====")
print("Exit code:", exit_code)
print("Log saved at:", RESUME_LOG)

if exit_code == 0:
    print("SUCCESS: YOLOv5 training completed normally.")
else:
    print("Training stopped before normal completion.")
    print("The checkpoint and console log remain available.")

[Output condensed for readability: progress bars and repeated warnings removed. The full console log is saved under experiments/YOLOv5/logs/]

Streaming output truncated to the last 5000 lines.

                   all       3444       4245      0.791      0.713      0.765      0.429

      Epoch    GPU_mem   box_loss   obj_loss   cls_loss  Instances       Size

                   all       3444       4245       0.79      0.711      0.765      0.429

25 epochs completed in 4.922 hours.
Optimizer stripped from /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/yolov5s_corrected_416_85/weights/last.pt, 14.3MB
Optimizer stripped from /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/yolov5s_corrected_416_85/weights/best.pt, 14.3MB

Validating /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/yolov5s_corrected_416_85/weights/best.pt...
Fusing layers...
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs

                   all       

### 5.3 Verify the final checkpoints

In [ ]:
from pathlib import Path
import os
import sys
import torch

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

YOLOV5_DIR = PROJECT / "external" / "YOLOv5"
RUN_DIR = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "yolov5s_corrected_416_85"
)

BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"

print("===== CHECKPOINT-LOADING PRECHECK =====")
print("YOLOv5 repository exists:", YOLOV5_DIR.exists())
print("models folder exists:", (YOLOV5_DIR / "models").exists())
print("best.pt exists:", BEST_WEIGHTS.exists())
print("last.pt exists:", LAST_WEIGHTS.exists())

required_paths = [
    YOLOV5_DIR,
    YOLOV5_DIR / "models",
    BEST_WEIGHTS,
    LAST_WEIGHTS,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    print("\nMissing paths:")

    for path in missing_paths:
        print(path)

    raise FileNotFoundError(
        "Checkpoint metadata could not be inspected."
    )

# Make YOLOv5's internal packages, including 'models', importable
if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))

# Use the repository as the current working directory
os.chdir(YOLOV5_DIR)

print("\nCurrent working directory:", Path.cwd())
print("YOLOv5 added to Python path:", str(YOLOV5_DIR) in sys.path)

print("\n===== CHECKPOINT METADATA =====")

for checkpoint_name, checkpoint_path in [
    ("best.pt", BEST_WEIGHTS),
    ("last.pt", LAST_WEIGHTS),
]:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    print(f"\n{checkpoint_name}")
    print("File size:", f"{checkpoint_path.stat().st_size / 1024**2:.2f} MB")
    print("Saved epoch:", checkpoint.get("epoch"))
    print("Best fitness:", checkpoint.get("best_fitness"))
    print("Optimizer included:", checkpoint.get("optimizer") is not None)
    print("Model included:", checkpoint.get("model") is not None)
    print("EMA model included:", checkpoint.get("ema") is not None)

    del checkpoint

print("\n===== CHECKPOINT RESULT =====")
print("SUCCESS: Both YOLOv5 checkpoints loaded correctly.")
print("The completed run is ready for held-out test evaluation.")

===== CHECKPOINT-LOADING PRECHECK =====
YOLOv5 repository exists: True
models folder exists: True
best.pt exists: True
last.pt exists: True

Current working directory: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5
YOLOv5 added to Python path: True

===== CHECKPOINT METADATA =====

best.pt
File size: 13.62 MB
Saved epoch: -1
Best fitness: None
Optimizer included: False
Model included: True
EMA model included: False

last.pt
File size: 13.62 MB
Saved epoch: -1
Best fitness: None
Optimizer included: False
Model included: True
EMA model included: False

===== CHECKPOINT RESULT =====
SUCCESS: Both YOLOv5 checkpoints loaded correctly.
The completed run is ready for held-out test evaluation.


## 6. Held-out test evaluation

### 6.1 mAP on the test set (confidence 0.001)

This run gives the reported mAP@0.5 and mAP@0.5:0.95.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

YOLOV5_DIR = PROJECT / "external" / "YOLOv5"
VAL_SCRIPT = YOLOV5_DIR / "val.py"

DATA_YAML = (
    PROJECT
    / "configs"
    / "YOLOv5"
    / "DFire_corrected.yaml"
)

BEST_WEIGHTS = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "yolov5s_corrected_416_85"
    / "weights"
    / "best.pt"
)

RESULTS_ROOT = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "test_evaluation"
)

LOG_DIR = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "logs"
)

print("===== TEST EVALUATION PRECHECK =====")

required_files = {
    "val.py": VAL_SCRIPT,
    "corrected dataset YAML": DATA_YAML,
    "best.pt": BEST_WEIGHTS,
}

all_available = True

for name, path in required_files.items():
    exists = path.exists()
    print(f"{name}: {path} -> {exists}")

    if not exists:
        all_available = False

if not all_available:
    raise FileNotFoundError(
        "A required evaluation file is missing. "
        "Test evaluation was not started."
    )

# Confirm that training is no longer running
process_check = subprocess.run(
    ["pgrep", "-af", "train.py"],
    text=True,
    capture_output=True,
)

print("\n===== ACTIVE TRAINING CHECK =====")

if process_check.returncode == 0:
    print(process_check.stdout.strip())
    raise RuntimeError(
        "A training process is still active. "
        "Test evaluation was not started."
    )
else:
    print("No active training process was found.")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_NAME = f"yolov5s_test_map_conf001_iou050_{timestamp}"
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

EVALUATION_LOG = (
    LOG_DIR
    / f"{RUN_NAME}.log"
)

command = [
    "python",
    "-u",
    str(VAL_SCRIPT),
    "--weights",
    str(BEST_WEIGHTS),
    "--data",
    str(DATA_YAML),
    "--img",
    "416",
    "--batch-size",
    "16",
    "--conf-thres",
    "0.001",
    "--iou-thres",
    "0.50",
    "--task",
    "test",
    "--device",
    "0",
    "--workers",
    "2",
    "--verbose",
    "--project",
    str(RESULTS_ROOT),
    "--name",
    RUN_NAME,
    "--exist-ok",
]

print("\n===== EVALUATION SETTINGS =====")
print("Model: YOLOv5s")
print("Weights:", BEST_WEIGHTS)
print("Dataset split: test")
print("Expected test images: 4306")
print("Image size: 416")
print("Batch size: 16")
print("Confidence threshold: 0.001")
print("NMS IoU threshold: 0.50")
print("Device: GPU 0")
print("Class-wise output: enabled")

print("\n===== EVALUATION COMMAND =====")
print(" ".join(command))

print("\nEvaluation log:")
print(EVALUATION_LOG)

print("\n===== STARTING HELD-OUT TEST EVALUATION =====\n")

with EVALUATION_LOG.open(
    "w",
    encoding="utf-8",
    buffering=1,
) as log_file:

    process = subprocess.Popen(
        command,
        cwd=YOLOV5_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

    exit_code = process.wait()

print("\n===== EVALUATION PROCESS ENDED =====")
print("Exit code:", exit_code)
print("Log saved at:", EVALUATION_LOG)
print("Results folder:", RESULTS_ROOT / RUN_NAME)

if exit_code == 0:
    print("SUCCESS: YOLOv5s held-out test evaluation completed.")
else:
    print("Evaluation did not complete normally.")
    print("Use the final log lines to identify the error.")

[Output condensed for readability: progress bars and repeated warnings removed. The full console log is saved under experiments/YOLOv5/logs/]

===== TEST EVALUATION PRECHECK =====
val.py: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5/val.py -> True
corrected dataset YAML: /content/drive/MyDrive/Fire_smoke_detection/configs/YOLOv5/DFire_corrected.yaml -> True
best.pt: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/yolov5s_corrected_416_85/weights/best.pt -> True

===== ACTIVE TRAINING CHECK =====
No active training process was found.

===== EVALUATION SETTINGS =====
Model: YOLOv5s
Weights: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/yolov5s_corrected_416_85/weights/best.pt
Dataset split: test
Expected test images: 4306
Image size: 416
Batch size: 16
Confidence threshold: 0.001
NMS IoU threshold: 0.50
Device: GPU 0
Class-wise output: enabled

===== EVALUATION COMMAND =====
python -u /content/drive/MyDrive/Fire_smoke_detection/external/YOLO

## 7. Operating-point metrics (confidence 0.25, IoU 0.50)

### 7.1 Generate predictions at confidence 0.25

Only the saved prediction files are used from this run. The mAP it prints is computed from predictions above
0.25 only, so it is not the reported mAP; that comes from 6.1.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

YOLOV5_DIR = PROJECT / "external" / "YOLOv5"
VAL_SCRIPT = YOLOV5_DIR / "val.py"

DATA_YAML = (
    PROJECT
    / "configs"
    / "YOLOv5"
    / "DFire_corrected.yaml"
)

BEST_WEIGHTS = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "yolov5s_corrected_416_85"
    / "weights"
    / "best.pt"
)

RESULTS_ROOT = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "test_evaluation"
)

LOG_DIR = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "logs"
)

print("===== FIXED-THRESHOLD EVALUATION PRECHECK =====")
print("val.py exists:", VAL_SCRIPT.exists())
print("Dataset YAML exists:", DATA_YAML.exists())
print("best.pt exists:", BEST_WEIGHTS.exists())

if not all([
    VAL_SCRIPT.exists(),
    DATA_YAML.exists(),
    BEST_WEIGHTS.exists(),
]):
    raise FileNotFoundError(
        "A required evaluation file is missing."
    )

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"yolov5s_test_conf025_iou050_{timestamp}"

OUTPUT_DIR = RESULTS_ROOT / RUN_NAME
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

EVALUATION_LOG = LOG_DIR / f"{RUN_NAME}.log"

command = [
    "python",
    "-u",
    str(VAL_SCRIPT),
    "--weights",
    str(BEST_WEIGHTS),
    "--data",
    str(DATA_YAML),
    "--img",
    "416",
    "--batch-size",
    "16",
    "--conf-thres",
    "0.25",
    "--iou-thres",
    "0.50",
    "--task",
    "test",
    "--device",
    "0",
    "--workers",
    "2",
    "--save-txt",
    "--save-conf",
    "--project",
    str(RESULTS_ROOT),
    "--name",
    RUN_NAME,
    "--exist-ok",
]

print("\n===== FIXED-THRESHOLD SETTINGS =====")
print("Test images: 4306")
print("Confidence threshold: 0.25")
print("NMS IoU threshold: 0.50")
print("Bounding-box matching IoU for next step: 0.50")
print("Saving detection confidence values: True")
print("Output directory:", OUTPUT_DIR)
print("Log:", EVALUATION_LOG)

print("\n===== STARTING PREDICTION GENERATION =====\n")

with EVALUATION_LOG.open(
    "w",
    encoding="utf-8",
    buffering=1,
) as log_file:

    process = subprocess.Popen(
        command,
        cwd=YOLOV5_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

    exit_code = process.wait()

print("\n===== PREDICTION GENERATION ENDED =====")
print("Exit code:", exit_code)
print("Output directory:", OUTPUT_DIR)
print("Prediction directory:", OUTPUT_DIR / "labels")
print("Log saved at:", EVALUATION_LOG)

prediction_files = (
    list((OUTPUT_DIR / "labels").glob("*.txt"))
    if (OUTPUT_DIR / "labels").exists()
    else []
)

print("Prediction files created:", len(prediction_files))

if exit_code == 0 and prediction_files:
    print("SUCCESS: Confidence-0.25 predictions are ready.")
else:
    print("Prediction generation needs inspection.")

[Output condensed for readability: progress bars and repeated warnings removed. The full console log is saved under experiments/YOLOv5/logs/]

===== FIXED-THRESHOLD EVALUATION PRECHECK =====
val.py exists: True
Dataset YAML exists: True
best.pt exists: True

===== FIXED-THRESHOLD SETTINGS =====
Test images: 4306
Confidence threshold: 0.25
NMS IoU threshold: 0.50
Bounding-box matching IoU for next step: 0.50
Saving detection confidence values: True
Output directory: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/test_evaluation/yolov5s_test_conf025_iou050_20260725_204135
Log: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/logs/yolov5s_test_conf025_iou050_20260725_204135.log

===== STARTING PREDICTION GENERATION =====

val: data=/content/drive/MyDrive/Fire_smoke_detection/configs/YOLOv5/DFire_corrected.yaml, weights=['/content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/yolov5s_corrected_416_85/weights/best.pt'], batch_size=16, imgsz=416, conf_th

### 7.2 Calculate exact TP / FP / FN, precision, recall and F1

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

DATASET = (
    PROJECT
    / "data"
    / "processed"
    / "DFireDataset_split"
)

TEST_IMAGES = DATASET / "images" / "test"
GROUND_TRUTH_DIR = DATASET / "labels" / "test"

EVALUATION_ROOT = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "test_evaluation"
)

# Find the most recent confidence-0.25 evaluation
evaluation_runs = sorted(
    EVALUATION_ROOT.glob(
        "yolov5s_test_conf025_iou050_*"
    ),
    key=lambda path: path.stat().st_mtime,
)

if not evaluation_runs:
    raise FileNotFoundError(
        "No confidence-0.25 evaluation directory was found."
    )

EVALUATION_DIR = evaluation_runs[-1]
PREDICTION_DIR = EVALUATION_DIR / "labels"

OUTPUT_CSV = (
    EVALUATION_DIR
    / "yolov5s_test_conf025_iou050_metrics.csv"
)

CLASS_NAMES = {
    0: "smoke",
    1: "fire",
}

IOU_THRESHOLD = 0.50
CONFIDENCE_THRESHOLD = 0.25

print("===== FIXED-THRESHOLD METRIC PRECHECK =====")
print("Evaluation directory:", EVALUATION_DIR)
print("Prediction directory:", PREDICTION_DIR)
print("Prediction directory exists:", PREDICTION_DIR.exists())
print("Ground-truth directory:", GROUND_TRUTH_DIR)
print("Ground-truth directory exists:", GROUND_TRUTH_DIR.exists())
print("Confidence threshold:", CONFIDENCE_THRESHOLD)
print("Matching IoU threshold:", IOU_THRESHOLD)

if not PREDICTION_DIR.exists():
    raise FileNotFoundError(
        "The prediction-label directory was not found."
    )

if not GROUND_TRUTH_DIR.exists():
    raise FileNotFoundError(
        "The ground-truth label directory was not found."
    )


def xywh_to_xyxy(box):
    """
    Convert a normalized YOLO box:
    x_center, y_center, width, height

    into:
    x_min, y_min, x_max, y_max
    """
    x_center, y_center, width, height = box

    return [
        x_center - width / 2,
        y_center - height / 2,
        x_center + width / 2,
        y_center + height / 2,
    ]


def calculate_iou(box_a, box_b):
    """
    Calculate intersection over union for two xyxy boxes.
    """
    intersection_x1 = max(box_a[0], box_b[0])
    intersection_y1 = max(box_a[1], box_b[1])
    intersection_x2 = min(box_a[2], box_b[2])
    intersection_y2 = min(box_a[3], box_b[3])

    intersection_width = max(
        0.0,
        intersection_x2 - intersection_x1,
    )
    intersection_height = max(
        0.0,
        intersection_y2 - intersection_y1,
    )

    intersection_area = (
        intersection_width * intersection_height
    )

    area_a = max(
        0.0,
        box_a[2] - box_a[0],
    ) * max(
        0.0,
        box_a[3] - box_a[1],
    )

    area_b = max(
        0.0,
        box_b[2] - box_b[0],
    ) * max(
        0.0,
        box_b[3] - box_b[1],
    )

    union_area = area_a + area_b - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area


def read_ground_truth(label_path):
    """
    Read one authoritative YOLO ground-truth label file.

    Expected row:
    class_id x_center y_center width height
    """
    annotations = []

    if not label_path.exists():
        raise FileNotFoundError(
            f"Missing ground-truth label: {label_path}"
        )

    for line_number, line in enumerate(
        label_path.read_text().splitlines(),
        start=1,
    ):
        line = line.strip()

        if not line:
            continue

        values = line.split()

        if len(values) != 5:
            raise ValueError(
                f"Invalid ground-truth row in {label_path}, "
                f"line {line_number}: {line}"
            )

        class_id = int(float(values[0]))
        coordinates = [float(value) for value in values[1:5]]

        annotations.append({
            "class_id": class_id,
            "box": xywh_to_xyxy(coordinates),
        })

    return annotations


def read_predictions(prediction_path):
    """
    Read one saved YOLOv5 prediction file.

    Expected row:
    class_id x_center y_center width height confidence
    """
    predictions = []

    # No file means that the image had no predictions
    if not prediction_path.exists():
        return predictions

    for line_number, line in enumerate(
        prediction_path.read_text().splitlines(),
        start=1,
    ):
        line = line.strip()

        if not line:
            continue

        values = line.split()

        if len(values) != 6:
            raise ValueError(
                f"Invalid prediction row in {prediction_path}, "
                f"line {line_number}: {line}"
            )

        class_id = int(float(values[0]))
        coordinates = [float(value) for value in values[1:5]]
        confidence = float(values[5])

        if confidence < CONFIDENCE_THRESHOLD:
            continue

        predictions.append({
            "class_id": class_id,
            "box": xywh_to_xyxy(coordinates),
            "confidence": confidence,
        })

    # Highest-confidence predictions are matched first
    return sorted(
        predictions,
        key=lambda item: item["confidence"],
        reverse=True,
    )


counts = {
    class_id: {
        "tp": 0,
        "fp": 0,
        "fn": 0,
        "ground_truth": 0,
        "predictions": 0,
    }
    for class_id in CLASS_NAMES
}

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
}

test_images = sorted(
    image for image in TEST_IMAGES.iterdir()
    if image.is_file()
    and image.suffix.lower() in image_extensions
)

print("\n===== DATASET CHECK =====")
print("Test images found:", len(test_images))
print("Expected test images: 4306")

if len(test_images) != 4306:
    raise RuntimeError(
        "The test image count is incorrect. "
        "Metric calculation was stopped."
    )

print("\n===== MATCHING PREDICTIONS TO GROUND TRUTH =====")

for image_number, image_path in enumerate(
    test_images,
    start=1,
):
    ground_truth_path = (
        GROUND_TRUTH_DIR
        / f"{image_path.stem}.txt"
    )

    prediction_path = (
        PREDICTION_DIR
        / f"{image_path.stem}.txt"
    )

    ground_truth = read_ground_truth(
        ground_truth_path
    )

    predictions = read_predictions(
        prediction_path
    )

    for class_id in CLASS_NAMES:
        class_ground_truth = [
            annotation
            for annotation in ground_truth
            if annotation["class_id"] == class_id
        ]

        class_predictions = [
            prediction
            for prediction in predictions
            if prediction["class_id"] == class_id
        ]

        counts[class_id]["ground_truth"] += len(
            class_ground_truth
        )

        counts[class_id]["predictions"] += len(
            class_predictions
        )

        matched_ground_truth = set()

        for prediction in class_predictions:
            best_iou = 0.0
            best_ground_truth_index = None

            for ground_truth_index, annotation in enumerate(
                class_ground_truth
            ):
                if ground_truth_index in matched_ground_truth:
                    continue

                iou = calculate_iou(
                    prediction["box"],
                    annotation["box"],
                )

                if iou > best_iou:
                    best_iou = iou
                    best_ground_truth_index = ground_truth_index

            if (
                best_ground_truth_index is not None
                and best_iou >= IOU_THRESHOLD
            ):
                counts[class_id]["tp"] += 1
                matched_ground_truth.add(
                    best_ground_truth_index
                )
            else:
                counts[class_id]["fp"] += 1

        unmatched_ground_truth = (
            len(class_ground_truth)
            - len(matched_ground_truth)
        )

        counts[class_id]["fn"] += unmatched_ground_truth

    if image_number % 500 == 0:
        print(
            f"Processed {image_number}/{len(test_images)} images"
        )

print(
    f"Processed {len(test_images)}/{len(test_images)} images"
)

print("\n===== EXACT OPERATING METRICS =====")

rows = []

for class_id, class_name in CLASS_NAMES.items():
    tp = counts[class_id]["tp"]
    fp = counts[class_id]["fp"]
    fn = counts[class_id]["fn"]

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    rows.append({
        "model": "YOLOv5s",
        "class_id": class_id,
        "class": class_name,
        "test_images": len(test_images),
        "ground_truth_objects": counts[class_id]["ground_truth"],
        "predictions": counts[class_id]["predictions"],
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    })

total_tp = sum(item["tp"] for item in counts.values())
total_fp = sum(item["fp"] for item in counts.values())
total_fn = sum(item["fn"] for item in counts.values())
total_ground_truth = sum(
    item["ground_truth"] for item in counts.values()
)
total_predictions = sum(
    item["predictions"] for item in counts.values()
)

overall_precision = (
    total_tp / (total_tp + total_fp)
    if total_tp + total_fp
    else 0.0
)

overall_recall = (
    total_tp / (total_tp + total_fn)
    if total_tp + total_fn
    else 0.0
)

overall_f1 = (
    2 * overall_precision * overall_recall
    / (overall_precision + overall_recall)
    if overall_precision + overall_recall
    else 0.0
)

rows.append({
    "model": "YOLOv5s",
    "class_id": -1,
    "class": "all",
    "test_images": len(test_images),
    "ground_truth_objects": total_ground_truth,
    "predictions": total_predictions,
    "true_positives": total_tp,
    "false_positives": total_fp,
    "false_negatives": total_fn,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "precision": overall_precision,
    "recall": overall_recall,
    "f1_score": overall_f1,
})

metrics = pd.DataFrame(rows)

display_columns = [
    "class",
    "ground_truth_objects",
    "predictions",
    "true_positives",
    "false_positives",
    "false_negatives",
    "precision",
    "recall",
    "f1_score",
]

print(
    metrics[display_columns].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

metrics.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("\n===== CONSISTENCY CHECKS =====")
print("Expected ground-truth objects: 5189")
print("Calculated ground-truth objects:", total_ground_truth)
print("TP + FN:", total_tp + total_fn)
print("Total saved predictions:", total_predictions)
print("TP + FP:", total_tp + total_fp)

print("\n===== SAVED RESULT =====")
print("Metrics CSV:", OUTPUT_CSV)
print("File exists:", OUTPUT_CSV.exists())

if (
    total_ground_truth == 5189
    and total_tp + total_fn == 5189
):
    print(
        "SUCCESS: Exact confidence-0.25 and "
        "IoU-0.50 metrics were calculated."
    )
else:
    print(
        "The totals require investigation before "
        "using these metrics."
    )

===== FIXED-THRESHOLD METRIC PRECHECK =====
Evaluation directory: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/test_evaluation/yolov5s_test_conf025_iou050_20260725_204135
Prediction directory: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/test_evaluation/yolov5s_test_conf025_iou050_20260725_204135/labels
Prediction directory exists: True
Ground-truth directory: /content/drive/MyDrive/Fire_smoke_detection/data/processed/DFireDataset_split/labels/test
Ground-truth directory exists: True
Confidence threshold: 0.25
Matching IoU threshold: 0.5

===== DATASET CHECK =====
Test images found: 4306
Expected test images: 4306

===== MATCHING PREDICTIONS TO GROUND TRUTH =====
Processed 500/4306 images
Processed 1000/4306 images
Processed 1500/4306 images
Processed 2000/4306 images
Processed 2500/4306 images
Processed 3000/4306 images
Processed 3500/4306 images
Processed 4000/4306 images
Processed 4306/4306 images

===== EXACT OPERATING METRICS =====
class  ground

### 7.3 Save the standardised YOLOv4 vs YOLOv5s comparison table

The YOLOv4 rows here are the values Darknet itself reported. Section 5.2 of `yolov4_test_evaluation.ipynb` later replaces the confidence-0.25 values for both models with ones from a common matching method and saves them as `yolov4_yolov5_test_metrics_standardized.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import math

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

COMPARISON_DIR = (
    PROJECT
    / "experiments"
    / "comparison"
)

COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_CSV = (
    COMPARISON_DIR
    / "yolov4_yolov5_test_metrics.csv"
)

print("===== STANDARDIZED COMPARISON SETTINGS =====")
print("Test images: 4306")
print("Ground-truth objects: 5189")
print("Classes: smoke and fire")
print("Input size: 416 × 416")
print("AP matching IoU: 0.50")
print("AP interpolation: 101 points")
print("Operating confidence threshold: 0.25")
print("Operating matching IoU: 0.50")

verified_results = [
    {
        "model": "YOLOv4",
        "class": "smoke",
        "test_images": 4306,
        "ground_truth_objects": 2311,
        "ap50": 0.8076,
        "map50_95": math.nan,
        "true_positives_conf025": 1823,
        "false_positives_conf025": 463,
        "false_negatives_conf025": 488,
    },
    {
        "model": "YOLOv4",
        "class": "fire",
        "test_images": 4306,
        "ground_truth_objects": 2878,
        "ap50": 0.6851,
        "map50_95": math.nan,
        "true_positives_conf025": 1985,
        "false_positives_conf025": 991,
        "false_negatives_conf025": 893,
    },
    {
        "model": "YOLOv4",
        "class": "all",
        "test_images": 4306,
        "ground_truth_objects": 5189,
        "ap50": 0.746340,
        "map50_95": math.nan,
        "true_positives_conf025": 3808,
        "false_positives_conf025": 1454,
        "false_negatives_conf025": 1381,
    },
    {
        "model": "YOLOv5s",
        "class": "smoke",
        "test_images": 4306,
        "ground_truth_objects": 2311,
        "ap50": 0.829,
        "map50_95": 0.490,
        "true_positives_conf025": 1914,
        "false_positives_conf025": 601,
        "false_negatives_conf025": 397,
    },
    {
        "model": "YOLOv5s",
        "class": "fire",
        "test_images": 4306,
        "ground_truth_objects": 2878,
        "ap50": 0.692,
        "map50_95": 0.344,
        "true_positives_conf025": 2043,
        "false_positives_conf025": 1319,
        "false_negatives_conf025": 835,
    },
    {
        "model": "YOLOv5s",
        "class": "all",
        "test_images": 4306,
        "ground_truth_objects": 5189,
        "ap50": 0.760,
        "map50_95": 0.417,
        "true_positives_conf025": 3957,
        "false_positives_conf025": 1920,
        "false_negatives_conf025": 1232,
    },
]

comparison = pd.DataFrame(verified_results)

comparison["confidence_threshold"] = 0.25
comparison["iou_threshold"] = 0.50

comparison["precision_conf025"] = (
    comparison["true_positives_conf025"]
    / (
        comparison["true_positives_conf025"]
        + comparison["false_positives_conf025"]
    )
)

comparison["recall_conf025"] = (
    comparison["true_positives_conf025"]
    / (
        comparison["true_positives_conf025"]
        + comparison["false_negatives_conf025"]
    )
)

comparison["f1_conf025"] = (
    2
    * comparison["precision_conf025"]
    * comparison["recall_conf025"]
    / (
        comparison["precision_conf025"]
        + comparison["recall_conf025"]
    )
)

comparison.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("\n===== SAVED COMPARISON TABLE =====")

display_columns = [
    "model",
    "class",
    "ap50",
    "map50_95",
    "precision_conf025",
    "recall_conf025",
    "f1_conf025",
    "true_positives_conf025",
    "false_positives_conf025",
    "false_negatives_conf025",
]

print(
    comparison[display_columns].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nComparison CSV:", OUTPUT_CSV)
print("File exists:", OUTPUT_CSV.exists())
print("Rows:", len(comparison))
print("Columns:", len(comparison.columns))

if OUTPUT_CSV.exists() and len(comparison) == 6:
    print(
        "SUCCESS: The verified test metrics were saved "
        "for final plotting and research reporting."
    )

===== STANDARDIZED COMPARISON SETTINGS =====
Test images: 4306
Ground-truth objects: 5189
Classes: smoke and fire
Input size: 416 × 416
AP matching IoU: 0.50
AP interpolation: 101 points
Operating confidence threshold: 0.25
Operating matching IoU: 0.50

===== SAVED COMPARISON TABLE =====
  model class     ap50  map50_95  precision_conf025  recall_conf025  f1_conf025  true_positives_conf025  false_positives_conf025  false_negatives_conf025
 YOLOv4 smoke 0.807600       NaN           0.797463        0.788836    0.793126                    1823                      463                      488
 YOLOv4  fire 0.685100       NaN           0.667003        0.689715    0.678169                    1985                      991                      893
 YOLOv4   all 0.746340       NaN           0.723679        0.733860    0.728734                    3808                     1454                     1381
YOLOv5s smoke 0.829000  0.490000           0.761034        0.828213    0.793203                

## 8. F1-confidence curve

### 8.1 Save predictions from confidence 0.001

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

YOLOV5_DIR = PROJECT / "external" / "YOLOv5"
VAL_SCRIPT = YOLOV5_DIR / "val.py"

DATA_YAML = (
    PROJECT
    / "configs"
    / "YOLOv5"
    / "DFire_corrected.yaml"
)

BEST_WEIGHTS = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "yolov5s_corrected_416_85"
    / "weights"
    / "best.pt"
)

RESULTS_ROOT = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "test_evaluation"
)

LOG_DIR = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "logs"
)

print("===== LOW-CONFIDENCE PREDICTION PRECHECK =====")
print("val.py exists:", VAL_SCRIPT.exists())
print("Dataset YAML exists:", DATA_YAML.exists())
print("best.pt exists:", BEST_WEIGHTS.exists())

if not all([
    VAL_SCRIPT.exists(),
    DATA_YAML.exists(),
    BEST_WEIGHTS.exists(),
]):
    raise FileNotFoundError(
        "A required evaluation file is missing."
    )

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"yolov5s_test_curve_conf001_iou050_{timestamp}"

OUTPUT_DIR = RESULTS_ROOT / RUN_NAME
EVALUATION_LOG = LOG_DIR / f"{RUN_NAME}.log"

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

command = [
    "python",
    "-u",
    str(VAL_SCRIPT),
    "--weights",
    str(BEST_WEIGHTS),
    "--data",
    str(DATA_YAML),
    "--img",
    "416",
    "--batch-size",
    "16",
    "--conf-thres",
    "0.001",
    "--iou-thres",
    "0.50",
    "--task",
    "test",
    "--device",
    "0",
    "--workers",
    "2",
    "--save-txt",
    "--save-conf",
    "--project",
    str(RESULTS_ROOT),
    "--name",
    RUN_NAME,
    "--exist-ok",
]

print("\n===== CURVE PREDICTION SETTINGS =====")
print("Test split: images/test")
print("Expected test images: 4306")
print("Minimum saved confidence: 0.001")
print("NMS IoU threshold: 0.50")
print("Image size: 416")
print("Weights: best.pt")
print("Output directory:", OUTPUT_DIR)
print("Log:", EVALUATION_LOG)

print("\n===== STARTING LOW-CONFIDENCE PREDICTIONS =====\n")

with EVALUATION_LOG.open(
    "w",
    encoding="utf-8",
    buffering=1,
) as log_file:

    process = subprocess.Popen(
        command,
        cwd=YOLOV5_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

    exit_code = process.wait()

print("\n===== LOW-CONFIDENCE PREDICTION PROCESS ENDED =====")
print("Exit code:", exit_code)
print("Output directory:", OUTPUT_DIR)
print("Prediction directory:", OUTPUT_DIR / "labels")
print("Log saved at:", EVALUATION_LOG)

prediction_files = (
    list((OUTPUT_DIR / "labels").glob("*.txt"))
    if (OUTPUT_DIR / "labels").exists()
    else []
)

prediction_rows = 0

for prediction_file in prediction_files:
    prediction_rows += sum(
        1
        for line in prediction_file.read_text().splitlines()
        if line.strip()
    )

print("Prediction files created:", len(prediction_files))
print("Total prediction rows saved:", prediction_rows)

if (
    exit_code == 0
    and prediction_files
    and prediction_rows > 0
):
    print(
        "SUCCESS: Low-confidence YOLOv5 predictions "
        "are ready for the F1-confidence curve."
    )
else:
    print(
        "The prediction output requires investigation."
    )

[Output condensed for readability: progress bars and repeated warnings removed. The full console log is saved under experiments/YOLOv5/logs/]

===== LOW-CONFIDENCE PREDICTION PRECHECK =====
val.py exists: True
Dataset YAML exists: True
best.pt exists: True

===== CURVE PREDICTION SETTINGS =====
Test split: images/test
Expected test images: 4306
Minimum saved confidence: 0.001
NMS IoU threshold: 0.50
Image size: 416
Weights: best.pt
Output directory: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/test_evaluation/yolov5s_test_curve_conf001_iou050_20260725_205411
Log: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/logs/yolov5s_test_curve_conf001_iou050_20260725_205411.log

===== STARTING LOW-CONFIDENCE PREDICTIONS =====

val: data=/content/drive/MyDrive/Fire_smoke_detection/configs/YOLOv5/DFire_corrected.yaml, weights=['/content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/yolov5s_corrected_416_85/weights/best.pt'], batch_size=16, imgsz=416, conf_t

### 8.2 Calculate the F1-confidence curve

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

DATASET = (
    PROJECT
    / "data"
    / "processed"
    / "DFireDataset_split"
)

TEST_IMAGES = DATASET / "images" / "test"
GROUND_TRUTH_DIR = DATASET / "labels" / "test"

EVALUATION_ROOT = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "test_evaluation"
)

COMPARISON_DIR = (
    PROJECT
    / "experiments"
    / "comparison"
)

COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_CSV = (
    COMPARISON_DIR
    / "yolov5s_f1_confidence_curve.csv"
)

CLASS_NAMES = {
    0: "smoke",
    1: "fire",
}

MATCHING_IOU = 0.50

print("===== F1-CURVE SOURCE CHECK =====")

curve_runs = sorted(
    EVALUATION_ROOT.glob(
        "yolov5s_test_curve_conf001_iou050_*"
    ),
    key=lambda path: path.stat().st_mtime,
)

if not curve_runs:
    raise FileNotFoundError(
        "No low-confidence YOLOv5 evaluation was found."
    )

EVALUATION_DIR = curve_runs[-1]
PREDICTION_DIR = EVALUATION_DIR / "labels"

print("Evaluation directory:", EVALUATION_DIR)
print("Prediction directory:", PREDICTION_DIR)
print("Prediction directory exists:", PREDICTION_DIR.exists())
print("Ground-truth directory exists:", GROUND_TRUTH_DIR.exists())

if not PREDICTION_DIR.exists():
    raise FileNotFoundError(
        "The saved prediction directory is missing."
    )


def xywh_to_xyxy(values):
    x_center, y_center, width, height = values

    return (
        x_center - width / 2,
        y_center - height / 2,
        x_center + width / 2,
        y_center + height / 2,
    )


def calculate_iou(box_a, box_b):
    intersection_x1 = max(box_a[0], box_b[0])
    intersection_y1 = max(box_a[1], box_b[1])
    intersection_x2 = min(box_a[2], box_b[2])
    intersection_y2 = min(box_a[3], box_b[3])

    intersection_width = max(
        0.0,
        intersection_x2 - intersection_x1,
    )

    intersection_height = max(
        0.0,
        intersection_y2 - intersection_y1,
    )

    intersection_area = (
        intersection_width * intersection_height
    )

    area_a = (
        max(0.0, box_a[2] - box_a[0])
        * max(0.0, box_a[3] - box_a[1])
    )

    area_b = (
        max(0.0, box_b[2] - box_b[0])
        * max(0.0, box_b[3] - box_b[1])
    )

    union_area = area_a + area_b - intersection_area

    return (
        intersection_area / union_area
        if union_area > 0
        else 0.0
    )


def read_ground_truth(path):
    annotations = []

    if not path.exists():
        raise FileNotFoundError(
            f"Missing ground-truth label: {path}"
        )

    for line_number, line in enumerate(
        path.read_text().splitlines(),
        start=1,
    ):
        line = line.strip()

        if not line:
            continue

        values = line.split()

        if len(values) != 5:
            raise ValueError(
                f"Invalid ground-truth row: "
                f"{path}, line {line_number}"
            )

        class_id = int(float(values[0]))
        box = xywh_to_xyxy(
            [float(value) for value in values[1:5]]
        )

        annotations.append({
            "class_id": class_id,
            "box": box,
        })

    return annotations


def read_predictions(path):
    predictions = []

    if not path.exists():
        return predictions

    for line_number, line in enumerate(
        path.read_text().splitlines(),
        start=1,
    ):
        line = line.strip()

        if not line:
            continue

        values = line.split()

        if len(values) != 6:
            raise ValueError(
                f"Invalid prediction row: "
                f"{path}, line {line_number}"
            )

        class_id = int(float(values[0]))
        box = xywh_to_xyxy(
            [float(value) for value in values[1:5]]
        )
        confidence = float(values[5])

        predictions.append({
            "class_id": class_id,
            "box": box,
            "confidence": confidence,
        })

    return sorted(
        predictions,
        key=lambda item: item["confidence"],
        reverse=True,
    )


image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
}

test_images = sorted(
    path for path in TEST_IMAGES.iterdir()
    if path.is_file()
    and path.suffix.lower() in image_extensions
)

print("\n===== TEST-SET CHECK =====")
print("Test images found:", len(test_images))
print("Expected test images: 4306")

if len(test_images) != 4306:
    raise RuntimeError(
        "The test-set image count is incorrect."
    )

# Store whether every confidence-ranked prediction
# is a TP or FP. This means matching is only done once.
prediction_outcomes = {
    class_id: []
    for class_id in CLASS_NAMES
}

ground_truth_counts = {
    class_id: 0
    for class_id in CLASS_NAMES
}

print("\n===== MATCHING ALL PREDICTIONS ONCE =====")

for image_number, image_path in enumerate(
    test_images,
    start=1,
):
    ground_truth = read_ground_truth(
        GROUND_TRUTH_DIR / f"{image_path.stem}.txt"
    )

    predictions = read_predictions(
        PREDICTION_DIR / f"{image_path.stem}.txt"
    )

    for class_id in CLASS_NAMES:
        class_ground_truth = [
            annotation
            for annotation in ground_truth
            if annotation["class_id"] == class_id
        ]

        class_predictions = [
            prediction
            for prediction in predictions
            if prediction["class_id"] == class_id
        ]

        ground_truth_counts[class_id] += len(
            class_ground_truth
        )

        matched_ground_truth = set()

        for prediction in class_predictions:
            best_iou = 0.0
            best_index = None

            for ground_truth_index, annotation in enumerate(
                class_ground_truth
            ):
                if ground_truth_index in matched_ground_truth:
                    continue

                iou = calculate_iou(
                    prediction["box"],
                    annotation["box"],
                )

                if iou > best_iou:
                    best_iou = iou
                    best_index = ground_truth_index

            is_true_positive = (
                best_index is not None
                and best_iou >= MATCHING_IOU
            )

            if is_true_positive:
                matched_ground_truth.add(best_index)

            prediction_outcomes[class_id].append({
                "confidence": prediction["confidence"],
                "is_true_positive": is_true_positive,
            })

    if image_number % 500 == 0:
        print(
            f"Processed {image_number}/{len(test_images)} images"
        )

print(
    f"Processed {len(test_images)}/{len(test_images)} images"
)

print("\n===== SOURCE TOTALS =====")

for class_id, class_name in CLASS_NAMES.items():
    print(
        f"{class_name}: "
        f"ground truth = {ground_truth_counts[class_id]}, "
        f"saved predictions = "
        f"{len(prediction_outcomes[class_id])}"
    )

print(
    "Total ground truth:",
    sum(ground_truth_counts.values()),
)

print(
    "Total saved predictions:",
    sum(
        len(items)
        for items in prediction_outcomes.values()
    ),
)

# Include 0.001, every hundredth from 0.01 to 0.99,
# the important 0.25 operating point, and 1.00.
thresholds = np.unique(
    np.concatenate([
        np.array([0.001]),
        np.arange(0.01, 1.00, 0.01),
        np.array([0.25, 1.00]),
    ])
)

curve_rows = []

print("\n===== CALCULATING CONFIDENCE CURVE =====")

for threshold in thresholds:
    class_metrics = {}

    for class_id, class_name in CLASS_NAMES.items():
        selected = [
            item
            for item in prediction_outcomes[class_id]
            if item["confidence"] >= threshold
        ]

        true_positives = sum(
            item["is_true_positive"]
            for item in selected
        )

        false_positives = (
            len(selected) - true_positives
        )

        false_negatives = (
            ground_truth_counts[class_id]
            - true_positives
        )

        precision = (
            true_positives
            / (true_positives + false_positives)
            if true_positives + false_positives
            else 0.0
        )

        recall = (
            true_positives
            / (true_positives + false_negatives)
            if true_positives + false_negatives
            else 0.0
        )

        f1_score = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall
            else 0.0
        )

        class_metrics[class_id] = {
            "tp": true_positives,
            "fp": false_positives,
            "fn": false_negatives,
        }

        curve_rows.append({
            "model": "YOLOv5s",
            "class": class_name,
            "confidence": float(threshold),
            "matching_iou": MATCHING_IOU,
            "true_positives": true_positives,
            "false_positives": false_positives,
            "false_negatives": false_negatives,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
        })

    overall_tp = sum(
        values["tp"]
        for values in class_metrics.values()
    )

    overall_fp = sum(
        values["fp"]
        for values in class_metrics.values()
    )

    overall_fn = sum(
        values["fn"]
        for values in class_metrics.values()
    )

    overall_precision = (
        overall_tp / (overall_tp + overall_fp)
        if overall_tp + overall_fp
        else 0.0
    )

    overall_recall = (
        overall_tp / (overall_tp + overall_fn)
        if overall_tp + overall_fn
        else 0.0
    )

    overall_f1 = (
        2 * overall_precision * overall_recall
        / (overall_precision + overall_recall)
        if overall_precision + overall_recall
        else 0.0
    )

    curve_rows.append({
        "model": "YOLOv5s",
        "class": "all",
        "confidence": float(threshold),
        "matching_iou": MATCHING_IOU,
        "true_positives": overall_tp,
        "false_positives": overall_fp,
        "false_negatives": overall_fn,
        "precision": overall_precision,
        "recall": overall_recall,
        "f1_score": overall_f1,
    })

curve = pd.DataFrame(curve_rows)

curve.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("Thresholds calculated:", len(thresholds))
print("CSV rows:", len(curve))

print("\n===== BEST F1 OPERATING POINTS =====")

for class_name in ["smoke", "fire", "all"]:
    class_curve = curve[
        curve["class"] == class_name
    ]

    best_index = class_curve["f1_score"].idxmax()
    best = class_curve.loc[best_index]

    print(
        f"{class_name}: "
        f"confidence = {best['confidence']:.3f}, "
        f"precision = {best['precision']:.6f}, "
        f"recall = {best['recall']:.6f}, "
        f"F1 = {best['f1_score']:.6f}"
    )

print("\n===== CONFIDENCE 0.25 VERIFICATION =====")

verification = curve[
    (curve["class"] == "all")
    & np.isclose(curve["confidence"], 0.25)
]

print(
    verification[
        [
            "confidence",
            "true_positives",
            "false_positives",
            "false_negatives",
            "precision",
            "recall",
            "f1_score",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nExpected confidence-0.25 values:")
print("TP: 3957")
print("FP: 1920")
print("FN: 1232")
print("Precision: 0.673303")
print("Recall: 0.762575")
print("F1: 0.715164")

print("\n===== SAVED CURVE =====")
print("Curve CSV:", OUTPUT_CSV)
print("File exists:", OUTPUT_CSV.exists())

if (
    OUTPUT_CSV.exists()
    and len(curve) == len(thresholds) * 3
):
    print(
        "SUCCESS: The YOLOv5s F1-confidence "
        "curve data was created."
    )
else:
    print(
        "The curve output requires investigation."
    )

===== F1-CURVE SOURCE CHECK =====
Evaluation directory: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/test_evaluation/yolov5s_test_curve_conf001_iou050_20260725_205411
Prediction directory: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/test_evaluation/yolov5s_test_curve_conf001_iou050_20260725_205411/labels
Prediction directory exists: True
Ground-truth directory exists: True

===== TEST-SET CHECK =====
Test images found: 4306
Expected test images: 4306

===== MATCHING ALL PREDICTIONS ONCE =====
Processed 500/4306 images
Processed 1000/4306 images
Processed 1500/4306 images
Processed 2000/4306 images
Processed 2500/4306 images
Processed 3000/4306 images
Processed 3500/4306 images
Processed 4000/4306 images
Processed 4306/4306 images

===== SOURCE TOTALS =====
smoke: ground truth = 2311, saved predictions = 23406
fire: ground truth = 2878, saved predictions = 30237
Total ground truth: 5189
Total saved predictions: 53643

===== CALCULATING CONFIDENCE CUR

## 9. Inference speed

### 9.1 Measure YOLOv5s latency

Batch 1, FP16, 10 warm-up images excluded, 200 measured images, timed with CUDA events. Uses the image lists
created by `yolov4_test_evaluation.ipynb` so both models are timed on the same images.

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import sys
import cv2
import numpy as np
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

YOLOV5_DIR = PROJECT / "external" / "YOLOv5"

BEST_WEIGHTS = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "yolov5s_corrected_416_85"
    / "weights"
    / "best.pt"
)

COMPARISON_DIR = (
    PROJECT
    / "experiments"
    / "comparison"
)

LOG_DIR = (
    PROJECT
    / "experiments"
    / "YOLOv5"
    / "logs"
)

WARMUP_LIST = (
    COMPARISON_DIR
    / "inference_benchmark_warmup_images.txt"
)

MEASURED_LIST = (
    COMPARISON_DIR
    / "inference_benchmark_measured_images.txt"
)

TIMINGS_CSV = (
    COMPARISON_DIR
    / "yolov5s_inference_timings.csv"
)

SUMMARY_CSV = (
    COMPARISON_DIR
    / "yolov5s_inference_summary.csv"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BENCHMARK_LOG = (
    LOG_DIR
    / f"yolov5s_inference_benchmark_{timestamp}.txt"
)

print("===== YOLOV5S SPEED PRECHECK =====")

required_files = {
    "YOLOv5 repository": YOLOV5_DIR,
    "Best weights": BEST_WEIGHTS,
    "Warm-up list": WARMUP_LIST,
    "Measured list": MEASURED_LIST,
}

all_available = True

for name, path in required_files.items():
    exists = path.exists()
    print(f"{name}: {path} -> {exists}")

    if not exists:
        all_available = False

if not all_available:
    raise FileNotFoundError(
        "A required YOLOv5s benchmark file is missing."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch cannot access the NVIDIA GPU."
    )

device = torch.device("cuda:0")
gpu_name = torch.cuda.get_device_name(0)

print("GPU:", gpu_name)
print("CUDA available:", torch.cuda.is_available())

warmup_paths = [
    line.strip()
    for line in WARMUP_LIST.read_text().splitlines()
    if line.strip()
]

measured_paths = [
    line.strip()
    for line in MEASURED_LIST.read_text().splitlines()
    if line.strip()
]

print("Warm-up paths:", len(warmup_paths))
print("Measured paths:", len(measured_paths))

if len(warmup_paths) != 10:
    raise RuntimeError(
        "Expected exactly 10 warm-up images."
    )

if len(measured_paths) != 200:
    raise RuntimeError(
        "Expected exactly 200 measured images."
    )

missing_images = [
    path
    for path in warmup_paths + measured_paths
    if not Path(path).exists()
]

print("Missing benchmark images:", len(missing_images))

if missing_images:
    raise FileNotFoundError(
        "Some benchmark images are unavailable."
    )

# Make the YOLOv5 repository's internal Python modules importable.
if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))

os.chdir(YOLOV5_DIR)

from models.experimental import attempt_load

try:
    from utils.augmentations import letterbox
except ImportError:
    # Compatibility fallback for older YOLOv5 releases.
    from utils.datasets import letterbox

print("\n===== LOADING YOLOV5S =====")

model = attempt_load(
    str(BEST_WEIGHTS),
    device=device,
)

model.eval()
model.half()

stride = int(
    model.stride.max()
    if hasattr(model, "stride")
    else 32
)

print("Weights:", BEST_WEIGHTS)
print("Device:", device)
print("Precision: FP16")
print("Model stride:", stride)

IMAGE_SIZE = 416


def prepare_tensor(image_path):
    """
    Load and letterbox one image to exactly 416 × 416.

    Image loading, resizing, normalization and GPU
    transfer are performed outside the timed region.
    """
    image = cv2.imread(str(image_path))

    if image is None:
        raise FileNotFoundError(
            f"OpenCV could not read: {image_path}"
        )

    image = letterbox(
        image,
        new_shape=(IMAGE_SIZE, IMAGE_SIZE),
        stride=stride,
        auto=False,
    )[0]

    image = image[:, :, ::-1].transpose(2, 0, 1)
    image = np.ascontiguousarray(image)

    tensor = torch.from_numpy(image).to(device)
    tensor = tensor.half()
    tensor /= 255.0
    tensor = tensor.unsqueeze(0)

    return tensor


print("\n===== YOLOV5S BENCHMARK SETTINGS =====")
print("GPU:", gpu_name)
print("Input tensor: 1 × 3 × 416 × 416")
print("Batch size: 1")
print("Precision: FP16")
print("Warm-up images excluded: 10")
print("Measured images: 200")
print("Preprocessing included in timing: False")
print("NMS included in timing: False")
print("Output saving: False")
print("Timing method: CUDA events")

print("\n===== GPU WARM-UP =====")

with torch.inference_mode():
    for index, image_path in enumerate(
        warmup_paths,
        start=1,
    ):
        tensor = prepare_tensor(image_path)

        _ = model(
            tensor,
            augment=False,
        )

        torch.cuda.synchronize()

        print(
            f"Warm-up {index}/{len(warmup_paths)} complete"
        )

print("\n===== MEASURING 200 IMAGES =====")

measured_times_ms = []
timing_rows = []

with torch.inference_mode():
    for index, image_path in enumerate(
        measured_paths,
        start=1,
    ):
        tensor = prepare_tensor(image_path)

        # Complete preprocessing and transfer before timing.
        torch.cuda.synchronize()

        start_event = torch.cuda.Event(
            enable_timing=True
        )

        end_event = torch.cuda.Event(
            enable_timing=True
        )

        start_event.record()

        _ = model(
            tensor,
            augment=False,
        )

        end_event.record()
        end_event.synchronize()

        latency_ms = float(
            start_event.elapsed_time(end_event)
        )

        measured_times_ms.append(latency_ms)

        timing_rows.append({
            "model": "YOLOv5s",
            "sequence_number": index,
            "image_name": Path(image_path).name,
            "image_path": image_path,
            "latency_ms": latency_ms,
        })

        if index % 25 == 0:
            print(
                f"Measured {index}/{len(measured_paths)} images"
            )

timings = pd.DataFrame(timing_rows)

timings.to_csv(
    TIMINGS_CSV,
    index=False,
)

mean_latency = float(
    np.mean(measured_times_ms)
)

median_latency = float(
    np.median(measured_times_ms)
)

standard_deviation = float(
    np.std(
        measured_times_ms,
        ddof=1,
    )
)

p95_latency = float(
    np.percentile(
        measured_times_ms,
        95,
    )
)

minimum_latency = float(
    np.min(measured_times_ms)
)

maximum_latency = float(
    np.max(measured_times_ms)
)

fps_from_mean_latency = (
    1000.0 / mean_latency
)

summary = pd.DataFrame([{
    "model": "YOLOv5s",
    "runtime": "PyTorch YOLOv5",
    "device": gpu_name,
    "input_width": 416,
    "input_height": 416,
    "batch_size": 1,
    "warmup_images": 10,
    "measured_images": 200,
    "confidence_threshold": 0.25,
    "timing_definition": "Native forward pass",
    "mean_latency_ms": mean_latency,
    "median_latency_ms": median_latency,
    "std_latency_ms": standard_deviation,
    "p95_latency_ms": p95_latency,
    "minimum_latency_ms": minimum_latency,
    "maximum_latency_ms": maximum_latency,
    "fps_from_mean_latency": fps_from_mean_latency,
}])

summary.to_csv(
    SUMMARY_CSV,
    index=False,
)

benchmark_text = f"""YOLOv5s controlled inference benchmark

Date: {datetime.now().isoformat()}
GPU: {gpu_name}
Weights: {BEST_WEIGHTS}
Input: 1 x 3 x 416 x 416
Precision: FP16
Warm-up images: 10
Measured images: 200
Preprocessing timed: False
NMS timed: False
Timing method: CUDA events

Mean latency: {mean_latency:.6f} ms
Median latency: {median_latency:.6f} ms
Standard deviation: {standard_deviation:.6f} ms
95th percentile: {p95_latency:.6f} ms
Minimum latency: {minimum_latency:.6f} ms
Maximum latency: {maximum_latency:.6f} ms
Estimated FPS: {fps_from_mean_latency:.6f}
"""

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BENCHMARK_LOG.write_text(
    benchmark_text
)

print("\n===== YOLOV5S SPEED RESULTS =====")
print(f"Mean latency:    {mean_latency:.3f} ms")
print(f"Median latency:  {median_latency:.3f} ms")
print(f"Std deviation:   {standard_deviation:.3f} ms")
print(f"95th percentile: {p95_latency:.3f} ms")
print(f"Minimum:         {minimum_latency:.3f} ms")
print(f"Maximum:         {maximum_latency:.3f} ms")
print(f"Estimated FPS:   {fps_from_mean_latency:.2f}")

print("\n===== SAVED YOLOV5S BENCHMARK =====")
print("Timings CSV:", TIMINGS_CSV)
print("Exists:", TIMINGS_CSV.exists())
print("Summary CSV:", SUMMARY_CSV)
print("Exists:", SUMMARY_CSV.exists())
print("Benchmark record:", BENCHMARK_LOG)
print("Exists:", BENCHMARK_LOG.exists())

if (
    len(timings) == 200
    and TIMINGS_CSV.exists()
    and SUMMARY_CSV.exists()
):
    print(
        "SUCCESS: YOLOv5s inference speed was measured "
        "on the controlled 200-image sample."
    )
else:
    print(
        "The YOLOv5s benchmark requires inspection."
    )

===== YOLOV5S SPEED PRECHECK =====
YOLOv5 repository: /content/drive/MyDrive/Fire_smoke_detection/external/YOLOv5 -> True
Best weights: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/yolov5s_corrected_416_85/weights/best.pt -> True
Warm-up list: /content/drive/MyDrive/Fire_smoke_detection/experiments/comparison/inference_benchmark_warmup_images.txt -> True
Measured list: /content/drive/MyDrive/Fire_smoke_detection/experiments/comparison/inference_benchmark_measured_images.txt -> True
GPU: Tesla T4
CUDA available: True
Warm-up paths: 10
Measured paths: 200
Missing benchmark images: 0

===== LOADING YOLOV5S =====
Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
Weights: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv5/yolov5s_corrected_416_85/weights/best.pt
Device: cuda:0
Precision: FP16
Model stride: 32

===== YOLOV5S BENCHMARK SETTINGS =====
GPU: Tesla T4
Input tensor: 1 × 3 × 416 × 416
Batch size: 1
Precision: FP1

### 9.2 Speed comparison chart (YOLOv4 vs YOLOv5s)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

COMPARISON_DIR = (
    PROJECT
    / "experiments"
    / "comparison"
)

FIGURE_DIR = COMPARISON_DIR / "figures"

YOLOV4_SUMMARY = (
    COMPARISON_DIR
    / "yolov4_inference_summary.csv"
)

YOLOV5_SUMMARY = (
    COMPARISON_DIR
    / "yolov5s_inference_summary.csv"
)

COMBINED_SUMMARY = (
    COMPARISON_DIR
    / "yolov4_yolov5_inference_comparison.csv"
)

OUTPUT_PNG = (
    FIGURE_DIR
    / "inference_speed_yolov4_yolov5.png"
)

OUTPUT_PDF = (
    FIGURE_DIR
    / "inference_speed_yolov4_yolov5.pdf"
)

print("===== SPEED-GRAPH PRECHECK =====")
print(
    "YOLOv4 summary:",
    YOLOV4_SUMMARY,
    "->",
    YOLOV4_SUMMARY.exists(),
)
print(
    "YOLOv5s summary:",
    YOLOV5_SUMMARY,
    "->",
    YOLOV5_SUMMARY.exists(),
)

if not YOLOV4_SUMMARY.exists():
    raise FileNotFoundError(
        "The YOLOv4 speed summary is missing."
    )

if not YOLOV5_SUMMARY.exists():
    raise FileNotFoundError(
        "The YOLOv5s speed summary is missing."
    )

yolov4 = pd.read_csv(YOLOV4_SUMMARY)
yolov5 = pd.read_csv(YOLOV5_SUMMARY)

if len(yolov4) != 1 or len(yolov5) != 1:
    raise RuntimeError(
        "Each speed-summary file must contain one row."
    )

speed = pd.concat(
    [
        yolov4,
        yolov5,
    ],
    ignore_index=True,
)

model_order = [
    "YOLOv4",
    "YOLOv5s",
]

speed = (
    speed
    .set_index("model")
    .loc[model_order]
    .reset_index()
)

required_columns = {
    "model",
    "mean_latency_ms",
    "median_latency_ms",
    "std_latency_ms",
    "p95_latency_ms",
    "fps_from_mean_latency",
    "measured_images",
    "batch_size",
}

missing_columns = required_columns - set(speed.columns)

if missing_columns:
    raise ValueError(
        f"Missing speed columns: "
        f"{sorted(missing_columns)}"
    )

print("\n===== CONTROLLED SPEED RESULTS =====")

display_columns = [
    "model",
    "mean_latency_ms",
    "median_latency_ms",
    "std_latency_ms",
    "p95_latency_ms",
    "fps_from_mean_latency",
]

print(
    speed[display_columns].to_string(
        index=False,
        float_format=lambda value: f"{value:.3f}",
    )
)

if not all(speed["measured_images"] == 200):
    raise RuntimeError(
        "Both models must use 200 measured images."
    )

if not all(speed["batch_size"] == 1):
    raise RuntimeError(
        "Both models must use batch size 1."
    )

yolov4_mean = float(
    speed.loc[
        speed["model"] == "YOLOv4",
        "mean_latency_ms",
    ].iloc[0]
)

yolov5_mean = float(
    speed.loc[
        speed["model"] == "YOLOv5s",
        "mean_latency_ms",
    ].iloc[0]
)

latency_speedup = (
    yolov4_mean / yolov5_mean
)

latency_reduction_percent = (
    (yolov4_mean - yolov5_mean)
    / yolov4_mean
    * 100
)

speed["latency_speedup_relative_to_yolov4"] = [
    1.0,
    latency_speedup,
]

speed["latency_reduction_vs_yolov4_percent"] = [
    0.0,
    latency_reduction_percent,
]

speed.to_csv(
    COMBINED_SUMMARY,
    index=False,
)

print("\n===== DERIVED COMPARISON =====")
print(
    f"YOLOv5s latency speedup: "
    f"{latency_speedup:.2f}×"
)
print(
    f"YOLOv5s mean-latency reduction: "
    f"{latency_reduction_percent:.2f}%"
)

print("\n===== CREATING SPEED FIGURE =====")

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model_colors = [
    "#1f77b4",
    "#ff7f0e",
]

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11, 5.2),
    constrained_layout=True,
)

x_positions = np.arange(len(model_order))

# Panel 1: Forward-pass latency
latency_bars = axes[0].bar(
    x_positions,
    speed["mean_latency_ms"],
    yerr=speed["std_latency_ms"],
    capsize=6,
    color=model_colors,
    edgecolor="white",
    linewidth=0.8,
)

axes[0].bar_label(
    latency_bars,
    labels=[
        f"{value:.2f} ms"
        for value in speed["mean_latency_ms"]
    ],
    padding=5,
    fontsize=10,
)

# Add p95 markers
axes[0].scatter(
    x_positions,
    speed["p95_latency_ms"],
    marker="D",
    s=55,
    color="black",
    label="95th percentile",
    zorder=5,
)

for position, p95_value in zip(
    x_positions,
    speed["p95_latency_ms"],
):
    axes[0].annotate(
        f"P95 {p95_value:.2f} ms",
        xy=(position, p95_value),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=8,
        ha="left",
    )

axes[0].set_title("Forward-pass latency")
axes[0].set_ylabel("Latency per image (ms)")
axes[0].set_xticks(
    x_positions,
    model_order,
)
axes[0].set_ylim(
    0,
    max(speed["p95_latency_ms"]) * 1.25,
)
axes[0].grid(
    axis="y",
    linestyle=":",
    linewidth=0.7,
    alpha=0.6,
)
axes[0].legend(
    loc="upper right",
    frameon=False,
)

# Panel 2: Estimated FPS
fps_bars = axes[1].bar(
    x_positions,
    speed["fps_from_mean_latency"],
    color=model_colors,
    edgecolor="white",
    linewidth=0.8,
)

axes[1].bar_label(
    fps_bars,
    labels=[
        f"{value:.1f} FPS"
        for value in speed["fps_from_mean_latency"]
    ],
    padding=5,
    fontsize=10,
)

axes[1].set_title("Estimated inference throughput")
axes[1].set_ylabel("Frames per second")
axes[1].set_xticks(
    x_positions,
    model_order,
)
axes[1].set_ylim(
    0,
    max(speed["fps_from_mean_latency"]) * 1.18,
)
axes[1].grid(
    axis="y",
    linestyle=":",
    linewidth=0.7,
    alpha=0.6,
)

axes[1].text(
    0.5,
    0.88,
    (
        f"YOLOv5s: {latency_speedup:.2f}× faster\n"
        f"{latency_reduction_percent:.1f}% lower mean latency"
    ),
    transform=axes[1].transAxes,
    ha="center",
    va="top",
    fontsize=10,
    bbox={
        "facecolor": "white",
        "edgecolor": "lightgray",
        "alpha": 0.9,
    },
)

fig.suptitle(
    (
        "Controlled inference-speed comparison\n"
        "Tesla T4, FP16, batch size 1, 416 × 416, "
        "200 identical test images"
    ),
    fontsize=13,
)

fig.text(
    0.5,
    -0.025,
    (
        "Timing covers the native neural-network "
        "forward pass only; preprocessing, NMS and "
        "output rendering are excluded."
    ),
    ha="center",
    va="top",
    fontsize=9,
)

fig.savefig(
    OUTPUT_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print("\n===== SAVED SPEED COMPARISON =====")
print("Combined CSV:", COMBINED_SUMMARY)
print("Exists:", COMBINED_SUMMARY.exists())
print("PNG:", OUTPUT_PNG)
print("Exists:", OUTPUT_PNG.exists())
print("PDF:", OUTPUT_PDF)
print("Exists:", OUTPUT_PDF.exists())

if (
    COMBINED_SUMMARY.exists()
    and OUTPUT_PNG.exists()
    and OUTPUT_PDF.exists()
):
    print(
        "SUCCESS: The controlled inference-speed "
        "comparison was completed."
    )
else:
    print(
        "One or more speed-comparison outputs are missing."
    )

===== SPEED-GRAPH PRECHECK =====
YOLOv4 summary: /content/drive/MyDrive/Fire_smoke_detection/experiments/comparison/yolov4_inference_summary.csv -> True
YOLOv5s summary: /content/drive/MyDrive/Fire_smoke_detection/experiments/comparison/yolov5s_inference_summary.csv -> True

===== CONTROLLED SPEED RESULTS =====
  model  mean_latency_ms  median_latency_ms  std_latency_ms  p95_latency_ms  fps_from_mean_latency
 YOLOv4           27.873             27.857           0.131          28.037                 35.877
YOLOv5s            8.300              7.591           2.104          11.546                120.476

===== DERIVED COMPARISON =====
YOLOv5s latency speedup: 3.36×
YOLOv5s mean-latency reduction: 70.22%

===== CREATING SPEED FIGURE =====

===== SAVED SPEED COMPARISON =====
Combined CSV: /content/drive/MyDrive/Fire_smoke_detection/experiments/comparison/yolov4_yolov5_inference_comparison.csv
Exists: True
PNG: /content/drive/MyDrive/Fire_smoke_detection/experiments/comparison/figures/infe